Data Preparation and Feature Engineering

Vehicle Valuation and Depreciation Intelligence Platform

Objective

The objective of this notebook is to prepare the used-vehicle listing data
for a baseline multiple linear regression model that predicts current listing
price.

The preparation process applies findings from exploratory data analysis,
including duplicate records, redundant columns, missing values, implausible
mileage observations, and high-cardinality categorical features.




In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn import datasets
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
DATA_PATH = Path("../data/processed/used_cars_reduced.csv")
df = pd.read_csv(DATA_PATH)

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

C:\Users\casey\AppData\Local\Temp\ipykernel_3260\950261239.py:8: DtypeWarning: Columns (0: dealer_zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [2]:
pd.set_option("display.max_columns", None)


In [3]:
"index" in df.columns

False

In [4]:
df

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
0,Bayamon,960,I4,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,NaN,FWD,2019
1,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
2,Guaynabo,969,H4,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,NaN,AWD,2016
3,San Juan,922,V6,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,NaN,AWD,2020
4,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,NaN,FWD,2018
2663247,Vallejo,94591,V6,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,NaN,FWD,2020
2663248,Napa,94559,NaN,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,NaN,FWD,2016
2663249,Fairfield,94533,I4 Diesel,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,NaN,AWD,2017


In [5]:
df.shape


(2663251, 21)

PHASE 1 i
Do new vehicles belong in the dataset?
Yes New vehicles provide the starting market reference before mileage, age, ownership, accidents, and other factors reduce value.


CHECK POINT 1. Target Price Validation

In [6]:
df['price'].isna()

0          False
1          False
2          False
3          False
4          False
           ...  
2663246    False
2663247    False
2663248    False
2663249    False
2663250    False
Name: price, Length: 2663251, dtype: bool

In [7]:
df['price'].isna().sum()

np.int64(0)

In [8]:
(df["price"] == 0).sum()

np.int64(0)

In [9]:
(df["price"] < 0).sum()

np.int64(0)

In [10]:
df['price'].max()
df.loc[df["price"] == df["price"].max()]

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1261170,Palm Harbor,34683,V12,V12,False,Gasoline,False,660.0,False,Ferrari,5339.0,Enzo,2.0,3299995.0,False,4.4,A,2 Dr STD Coupe,NaN,RWD,2003


In [11]:
df['price'].min()
df.loc[df["price"] == df["price"].min()]

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1320599,Belle Glade,33430,V6,V6,False,Gasoline,False,160.0,False,Buick,190000.0,Century,4.0,165.0,False,3.565217,A,Custom Sedan FWD,NaN,FWD,1999


In [12]:
df['mileage'] 

0              7.0
1              8.0
2              NaN
3             11.0
4              7.0
            ...   
2663246    41897.0
2663247        5.0
2663248    57992.0
2663249    27857.0
2663250    22600.0
Name: mileage, Length: 2663251, dtype: float64

In [13]:

comparison=df.loc[(df["make_name"]== 'Buick') & 
                  (df["model_name"] == "Century"),
[       
        "year",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        "price"
   ]
]
comparison.shape
comparison.sort_values("price").head(20)

,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1320599,1999,190000.0,4.0,False,False,False,165.0
878365,2005,202158.0,5.0,True,False,False,249.0
301631,2001,175689.0,2.0,False,False,False,495.0
1148231,2002,NaN,5.0,False,False,False,600.0
1209120,2002,150000.0,4.0,False,False,False,750.0
1687931,2003,160730.0,4.0,True,False,False,995.0
1148579,1999,130474.0,3.0,True,False,False,1400.0
1738903,2002,236760.0,4.0,True,False,False,1490.0
44862,2001,185524.0,2.0,False,False,False,1499.0
1523587,2004,165385.0,5.0,False,False,False,1500.0


In [14]:
df["frame_damaged"].any()

np.True_

In [15]:
df.sort_values("price").head(5)

,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
1320599,Belle Glade,33430,V6,V6,False,Gasoline,False,160.0,False,Buick,190000.0,Century,4.0,165.0,False,3.565217,A,Custom Sedan FWD,NaN,FWD,1999
1359718,Medley,33178,NaN,NaN,False,NaN,False,210.0,False,Ford,150000.0,Explorer,4.0,200.0,False,4.000000,A,XLT V6,NaN,RWD,2005
878365,Traverse City,49684,V6,V6,False,Gasoline,True,175.0,False,Buick,202158.0,Century,5.0,249.0,False,3.593750,A,Custom Sedan FWD,NaN,FWD,2005
1271928,Bainbridge,39817,NaN,NaN,False,NaN,True,170.0,False,Nissan,NaN,Altima Coupe,5.0,250.0,False,1.000000,CVT,2.5 S,NaN,FWD,2008
1271923,Bainbridge,39817,V6,V6,False,Gasoline,False,290.0,False,Acura,NaN,RL,3.0,250.0,False,1.000000,A,SH-AWD with Navigation and Tech Package,NaN,AWD,2006


In [16]:
(df["price"] < 500).sum()

np.int64(86)

In [17]:
(df["price"] < 1000).sum()

np.int64(494)

In [18]:
df.loc[
    df["price"] < 1000,
    [
        "make_name",
        "model_name",
        "year",
        "is_new",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,is_new,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1320599,Buick,Century,1999,False,190000.0,4.0,False,False,False,165.0
1359718,Ford,Explorer,2005,False,150000.0,4.0,False,False,False,200.0
878365,Buick,Century,2005,False,202158.0,5.0,True,False,False,249.0
1271941,Kia,Sorento,2006,False,NaN,6.0,True,False,False,250.0
1271923,Acura,RL,2006,False,NaN,3.0,False,False,False,250.0
1271928,Nissan,Altima Coupe,2008,False,NaN,5.0,True,False,False,250.0
1271918,Toyota,Avalon,2005,False,NaN,5.0,True,False,False,250.0
1271913,Mitsubishi,Lancer,2008,False,NaN,2.0,True,False,False,250.0
1938898,Mercury,Villager,1998,False,104000.0,3.0,False,False,False,256.0
1579657,Pontiac,Vibe,2003,False,200000.0,5.0,True,False,False,299.0


In [19]:
comparison_Nissan=df.loc[(df["make_name"]== 'Nissan') & 
                  (df["model_name"] == "Altima"),
[       
        "year",
        "mileage",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        "price"
   ]
]
comparison_Nissan.shape
comparison_Nissan.sort_values("price").head(20)

,year,mileage,owner_count,has_accidents,frame_damaged,salvage,price
1394034,2020,NaN,1.0,False,False,False,386.9
1320684,2004,175222.0,7.0,True,False,False,484.0
1320691,1998,177420.0,2.0,False,False,False,484.0
949965,1997,NaN,3.0,True,False,False,550.0
1579680,2006,204952.0,4.0,True,False,False,677.0
1677446,2005,200829.0,6.0,True,False,False,800.0
1250485,2003,191652.0,NaN,False,False,False,899.0
950307,2002,NaN,4.0,False,False,False,900.0
559459,2003,NaN,2.0,True,False,False,900.0
950193,1997,NaN,4.0,True,False,False,900.0


In [20]:
df.loc[df["price"] < 1000, "price"].value_counts()

price
999.0    95
995.0    64
900.0    32
484.0    27
495.0    23
         ..
389.0     1
980.0     1
256.0     1
890.0     1
449.0     1
Name: count, Length: 72, dtype: int64

In [21]:
df.loc[
    df["price"] == 999,
    [
        "make_name",
        "model_name",
        "year",
        "mileage",
        "city",
        "is_new",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        'dealer_zip',
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,city,is_new,owner_count,has_accidents,frame_damaged,salvage,dealer_zip,price
2686,Honda,Civic,1996,222434.0,Teterboro,False,4.0,True,False,False,7608,999.0
2690,Nissan,Maxima,2001,NaN,Teterboro,False,3.0,True,False,False,7608,999.0
126057,Nissan,Sentra,2005,205000.0,East Granby,False,2.0,False,False,False,6026,999.0
151782,Mercedes-Benz,420-Class,1987,203189.0,New Windsor,False,2.0,False,False,False,12553,999.0
158302,Ford,Focus,2002,231000.0,Farmington,False,4.0,True,False,False,55024,999.0
158318,INFINITI,I30,1997,320058.0,Spanaway,False,6.0,False,False,False,98387,999.0
166047,Mazda,MAZDA6,2003,175000.0,Manchester,False,6.0,False,False,False,03103,999.0
326052,Ford,Taurus,1997,104537.0,Edison,False,2.0,False,False,False,8817,999.0
481932,Mercury,Grand Marquis,1999,177832.0,Mchenry,False,4.0,True,False,False,60051,999.0
527526,Kia,Spectra,2004,NaN,Peninsula,False,4.0,False,False,False,44264,999.0


In [22]:
df.loc[df["price"] == 999, "dealer_zip"].value_counts(dropna=False)

dealer_zip
32211    44
55024     7
55906     3
7608      2
56301     2
57401     2
84054     2
6026      1
12553     1
55024     1
98387     1
03103     1
8817      1
60051     1
44264     1
60636     1
22191     1
20748     1
49058     1
37601     1
30458     1
32714     1
34748     1
50401     1
33619     1
33771     1
33168     1
56258     1
68701     1
41042     1
56387     1
84070     1
84095     1
59601     1
84047     1
72764     1
66720     1
68901     1
89048     1
91786     1
Name: count, dtype: int64

In [23]:
df.loc[
    df["price"] < 999,
    [
        "make_name",
        "model_name",
        "year",
        "mileage",
        "city",
        "is_new",
        "owner_count",
        "has_accidents",
        "frame_damaged",
        "salvage",
        'dealer_zip',
        "price"
    ]
].sort_values("price").head(30)

,make_name,model_name,year,mileage,city,is_new,owner_count,has_accidents,frame_damaged,salvage,dealer_zip,price
1320599,Buick,Century,1999,190000.0,Belle Glade,False,4.0,False,False,False,33430,165.0
1359718,Ford,Explorer,2005,150000.0,Medley,False,4.0,False,False,False,33178,200.0
878365,Buick,Century,2005,202158.0,Traverse City,False,5.0,True,False,False,49684,249.0
1271923,Acura,RL,2006,NaN,Bainbridge,False,3.0,False,False,False,39817,250.0
1271913,Mitsubishi,Lancer,2008,NaN,Bainbridge,False,2.0,True,False,False,39817,250.0
1271928,Nissan,Altima Coupe,2008,NaN,Bainbridge,False,5.0,True,False,False,39817,250.0
1271918,Toyota,Avalon,2005,NaN,Bainbridge,False,5.0,True,False,False,39817,250.0
1271941,Kia,Sorento,2006,NaN,Bainbridge,False,6.0,True,False,False,39817,250.0
1938898,Mercury,Villager,1998,104000.0,Henryetta,False,3.0,False,False,False,74437,256.0
141899,Chrysler,Town & Country,2007,NaN,Mount Morris,False,1.0,False,False,False,48458,299.0


In [24]:
df.loc[df["price"] < 999, "dealer_zip"].value_counts(dropna=False)

dealer_zip
29073    83
60914    31
34266    11
33430    10
33935    10
         ..
81082     1
89801     1
80204     1
80920     1
81003     1
Name: count, Length: 168, dtype: int64

CHECK POINT 1 COMPLETE
Target Price Checkpoint

- `price` has no missing, zero, or negative values.
- 494 listings are priced below $1,000.
- Repeated prices such as $999 appear across substantially different makes, models, years, mileage, and condition histories
- Many ultra-low prices are concentrated in specific dealer ZIP codes, suggesting placeholder or promotional pricing.

Current Conclusion

The repeated ultra-low prices across dissimilar vehicles, combined with their
concentration in specific dealer locations, suggest that some values below
$1,000 may represent dealer-specific pricing conventions, placeholder prices,
deposits, or other values that do not reflect full vehicle market value.

These records have not yet been removed from the original dataset. The current
proposed policy is to exclude listings priced below $1,000 from the dataset used
to train the initial current-value model while preserving them in the original
data.

In [25]:
valid_prices = df.loc[df['price'] >= 1000].copy()


In [26]:
valid_prices.shape

(2662757, 21)

In [27]:
valid_prices



,city,dealer_zip,engine_cylinders,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,vehicle_damage_category,wheel_system,year
0,Bayamon,960,I4,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,NaN,FWD,2019
1,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
2,Guaynabo,969,H4,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,NaN,AWD,2016
3,San Juan,922,V6,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,NaN,AWD,2020
4,San Juan,922,I4,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,NaN,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,NaN,FWD,2018
2663247,Vallejo,94591,V6,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,NaN,FWD,2020
2663248,Napa,94559,NaN,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,NaN,FWD,2016
2663249,Fairfield,94533,I4 Diesel,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,NaN,AWD,2017


The major players
Price — the target we must protect from bad labels.
Vehicle identity — make, model, trim, year.
Lifecycle — is_new, mileage, owner count.
Condition — accidents, frame damage, salvage.
Market context — city, dealer ZIP, seller rating.

The central relationship is:

Vehicle identity sets the expected value range; lifecycle and condition adjust it; market context may shift the listing price.

Consistency checks

Instead of inventing arbitrary mileage cutoffs, we look for conflicts:

New vehicle + extremely high mileage
Used vehicle + zero mileage
New vehicle + several owners
Old vehicle + marked new
Price wildly inconsistent with similar vehicles
Repeated prices concentrated in one dealer location

In [28]:
valid_prices["price"].min()

np.float64(1000.0)

In [29]:
valid_prices.drop(columns=["vehicle_damage_category"], inplace=True)

In [30]:
valid_prices.drop_duplicates(inplace=True)

In [31]:
valid_prices.info()

<class 'pandas.DataFrame'>
Index: 2662757 entries, 0 to 2663250
Data columns (total 20 columns):
 #   Column            Dtype  
---  ------            -----  
 0   city              str    
 1   dealer_zip        object 
 2   engine_cylinders  str    
 3   engine_type       str    
 4   frame_damaged     object 
 5   fuel_type         str    
 6   has_accidents     object 
 7   horsepower        float64
 8   is_new            bool   
 9   make_name         str    
 10  mileage           float64
 11  model_name        str    
 12  owner_count       float64
 13  price             float64
 14  salvage           object 
 15  seller_rating     float64
 16  transmission      str    
 17  trim_name         str    
 18  wheel_system      str    
 19  year              int64  
dtypes: bool(1), float64(5), int64(1), object(4), str(9)
memory usage: 543.6+ MB


In [32]:
valid_prices["engine_type"].equals (valid_prices["engine_cylinders"])

True

In [33]:
valid_prices.drop(columns =["engine_cylinders"], inplace = True)

In [34]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663247,Vallejo,94591,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,FWD,2020
2663248,Napa,94559,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


CHECKPOINT 2 Mileage validation

In [35]:
valid_prices.loc[
    valid_prices["mileage"] >= 1_000_000,
    [
        "mileage",
        "year",
        "make_name",
        "model_name",
        "price",
        "is_new",
        "city",
        "engine_type",
        "owner_count"
    ]
].sort_values("mileage")

,mileage,year,make_name,model_name,price,is_new,city,engine_type,owner_count
1853098,1111111.0,2020,Chevrolet,Equinox,23620.0,True,Boise,I4,NaN
1853005,1111111.0,2020,Chevrolet,Silverado 1500,39523.0,True,Boise,V8,NaN
1855598,1111111.0,2020,Cadillac,XT5,45970.0,True,Boise,I4,NaN
1853986,1111111.0,2020,Chevrolet,Silverado 1500,41420.0,True,Boise,V8,NaN
1180953,1225238.0,2020,Ford,F-150,45149.0,True,Belleview,V6,NaN
185063,4290461.0,2020,Chevrolet,Silverado 1500,50775.0,True,Nappanee,V8,NaN
2242507,99999988.0,2019,RAM,3500 Chassis,52610.0,True,Houston,I6 Diesel,NaN


In [36]:
valid_prices.loc[
    (
        (valid_prices["make_name"] == "Chevrolet") & (valid_prices["model_name"] == "Equinox")
    ) | (
        (valid_prices["make_name"] == "Chevrolet") & (valid_prices["model_name"] == "Silverado 1500")
    ),
    [ "make_name", "model_name", "mileage", "year", "price", "is_new", "city", "engine_type", "owner_count"]
].sort_values('mileage').head(50)

,make_name,model_name,mileage,year,price,is_new,city,engine_type,owner_count
1483414,Chevrolet,Equinox,0.0,2020,21801.0,True,Bowling Green,I4,NaN
1483379,Chevrolet,Equinox,0.0,2020,21372.0,True,Bowling Green,I4,NaN
2135402,Chevrolet,Silverado 1500,0.0,2020,27317.0,True,Georgetown,V8,NaN
2135367,Chevrolet,Silverado 1500,0.0,2020,48315.0,True,Weimar,V8,NaN
1769178,Chevrolet,Silverado 1500,0.0,2020,56400.0,True,Havre,V8,NaN
1769076,Chevrolet,Silverado 1500,0.0,2020,68035.0,True,Havre,V8,NaN
1768813,Chevrolet,Silverado 1500,0.0,2020,50995.0,True,Conrad,V8,NaN
1000866,Chevrolet,Silverado 1500,0.0,2020,39645.0,True,Lawrenceville,I6 Diesel,NaN
1000602,Chevrolet,Silverado 1500,0.0,2020,36205.0,True,Lawrenceville,V8,NaN
1000576,Chevrolet,Equinox,0.0,2020,31330.0,True,Lawrenceville,I4,NaN


In [37]:
valid_prices.loc[
    valid_prices["mileage"] >= 1_000_000,
    "mileage"
] = np.nan

In [38]:
(valid_prices["mileage"] < 0).sum()

np.int64(0)

In [39]:
valid_prices.loc[
    valid_prices["mileage"] >= 999_999,
    "mileage"
] = np.nan

In [40]:
valid_prices.nlargest(10, "mileage")[
    ["make_name", "model_name", "year", "mileage", "price", "is_new"]
]

,make_name,model_name,year,mileage,price,is_new
56105,Buick,Encore,2020,785778.0,28230.0,True
2464582,Ford,F-150,2020,631835.0,51427.0,True
2454610,Jeep,Grand Cherokee,2020,610288.0,42763.0,True
1596466,Chevrolet,Silverado 3500HD Chassis,2020,590072.0,45288.0,True
451359,Chevrolet,Equinox,2020,514033.0,27999.0,True
203429,Jeep,Cherokee,2020,434718.0,36585.0,True
755125,Ford,F-250 Super Duty,2001,400000.0,5995.0,False
1186639,Ford,F-350 Super Duty,2004,399900.0,9800.0,False
2182925,Dodge,RAM 2500,2007,399578.0,9900.0,False
1435171,Lexus,ES 350,2008,399496.0,6259.0,False


In [41]:
valid_prices.loc[
    valid_prices["mileage"] == 0,  "is_new"
].value_counts(dropna = False)

is_new
True     178969
False         3
Name: count, dtype: int64

In [42]:
valid_prices.loc[(valid_prices["mileage"] == 0) & (valid_prices["is_new"] == False)]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
1035593,Charleston,29414,I3,False,Gasoline,False,78.0,False,Mitsubishi,0.0,Mirage G4,2.0,16354.7,False,3.615385,M,ES FWD,FWD,2018
1446904,Lansing,66043,V6 Biodiesel,False,Biodiesel,False,260.0,False,RAM,0.0,1500,NaN,46166.0,False,5.000000,A,Big Horn Crew Cab 4WD,4WD,2020
1991260,Crosby,77532,V6,False,Gasoline,False,450.0,False,Ford,0.0,F-150,NaN,60609.0,False,3.800000,A,SVT Raptor SuperCrew 4WD,4WD,2020


In [43]:
#Zero mileage is retained for new vehicles, but zero mileage on used vehicles is treated as missing because it is not credible as an odometer reading.
valid_prices.loc[
    (valid_prices["mileage"] == 0) &
    (valid_prices["is_new"] == False),
    "mileage"
] = np.nan

valid_prices.loc[
    (valid_prices["mileage"] == 0)&
    (valid_prices["is_new"] == False)
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year


In [44]:
valid_prices["mileage"].max()

np.float64(785778.0)

In [45]:
valid_prices.loc[
    valid_prices["mileage"] == valid_prices["mileage"].max()
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
56105,Englewood,7631,I4,NaN,Gasoline,NaN,138.0,True,Buick,785778.0,Encore,NaN,28230.0,NaN,4.285714,A,Preferred AWD,AWD,2020


In [46]:
valid_prices.loc[
    (valid_prices["mileage"] >= 100000)&
    (valid_prices["is_new"] == True)
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
56105,Englewood,7631,I4,NaN,Gasoline,NaN,138.0,True,Buick,785778.0,Encore,NaN,28230.0,NaN,4.285714,A,Preferred AWD,AWD,2020
57037,Englewood,7631,V6,NaN,Gasoline,NaN,308.0,True,Chevrolet,115111.0,Blazer,NaN,39260.0,NaN,4.285714,A,1LT AWD,AWD,2020
58046,Englewood,7631,V6,NaN,Gasoline,NaN,310.0,True,Buick,381519.0,Enclave,NaN,45765.0,NaN,4.285714,A,Essence AWD,4WD,2020
68505,Woburn,1801,I4,NaN,Gasoline,NaN,170.0,True,Chevrolet,104261.0,Equinox,NaN,32340.0,NaN,4.714286,A,1.5T LT AWD,4WD,2020
203429,Portsmouth,3801,V6,NaN,Gasoline,NaN,271.0,True,Jeep,434718.0,Cherokee,NaN,36585.0,NaN,4.444444,A,Limited 4WD,4WD,2020
428139,Tunkhannock,18657,V8,NaN,Gasoline,NaN,420.0,True,Chevrolet,149304.0,Silverado 1500,NaN,55200.0,NaN,4.833333,A,LTZ Crew Cab 4WD,4WD,2020
451359,Bath,14810,I4,NaN,Gasoline,NaN,170.0,True,Chevrolet,514033.0,Equinox,NaN,27999.0,NaN,4.562500,A,2.0T LT AWD,4WD,2020
750379,Cary,27511,I4,NaN,Gasoline,NaN,170.0,True,Nissan,139592.0,Rogue,NaN,27965.0,NaN,4.320000,A,S FWD,FWD,2020
980582,Mc Donald,37353,I4,NaN,Gasoline,NaN,141.0,True,Nissan,118335.0,Rogue Sport,NaN,24454.0,NaN,4.882353,CVT,SV FWD,FWD,2020
1045927,Charleston,29407,I4,NaN,Gasoline,NaN,170.0,True,Nissan,166923.0,Rogue,NaN,33116.0,NaN,3.680000,CVT,SL FWD,FWD,2019


In [47]:
valid_prices.groupby("is_new")["mileage"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
)[["min", "50%", "90%", "95%", "99%", "max"]]

,min,50%,90%,95%,99%,max
is_new,,,,,,
False,1.0,41168.0,127764.3,154689.15,208780.0,400000.0
True,0.0,6.0,36.0,287.00,4769.0,785778.0


In [48]:
valid_prices.loc[
    valid_prices["is_new"] == True
].nlargest(20, "mileage")[
    ["make_name", "model_name", "year", "mileage", "price", "owner_count", ]
]

,make_name,model_name,year,mileage,price,owner_count
56105,Buick,Encore,2020,785778.0,28230.0,NaN
2464582,Ford,F-150,2020,631835.0,51427.0,NaN
2454610,Jeep,Grand Cherokee,2020,610288.0,42763.0,NaN
1596466,Chevrolet,Silverado 3500HD Chassis,2020,590072.0,45288.0,NaN
451359,Chevrolet,Equinox,2020,514033.0,27999.0,NaN
203429,Jeep,Cherokee,2020,434718.0,36585.0,NaN
58046,Buick,Enclave,2020,381519.0,45765.0,NaN
2440543,Ford,EcoSport,2020,360012.0,21499.0,NaN
2144275,Chevrolet,Camaro,2020,351123.0,26155.0,NaN
2660833,Chrysler,Pacifica Hybrid,2020,328727.0,42155.0,NaN


In [49]:
((valid_prices["is_new"] == True) & (valid_prices["mileage"] > 10_000)).sum()

np.int64(1348)

In [50]:
((valid_prices["is_new"] == True) & (valid_prices["mileage"] > 50_000)).sum()

np.int64(114)

In [51]:
((valid_prices["is_new"] == True) & (valid_prices["mileage"] > 100_000)).sum()

np.int64(29)

In [52]:
valid_prices.loc[
    (valid_prices['is_new'] == True) &
    (valid_prices['mileage'] > 50000),
    'mileage'
] =np.nan

valid_prices.loc[
    (valid_prices["is_new"] == True)&
    (valid_prices["mileage"] > 50000),
    'mileage'
].sum()

np.float64(0.0)

CHECK POINT 2 COMPLETE 
Mileage validation

Extreme placeholder mileages were changed to NaN.

Zero mileage was retained for new vehicles.

Zero mileage on used vehicles was changed to NaN.

Mileage above 50,000 on new vehicles was changed to NaN.

Vehicle rows were preserved.

CHECK POINT 3---> Owner-count validation
Vehicles marked as new should normally have no previous owners recorded.

In [53]:
valid_prices.groupby("is_new")["owner_count"].describe()[
    ["count", "min", "50%", "max"]
]

,count,min,50%,max
is_new,,,,
False,1480648.0,1.0,1.0,19.0
True,965.0,1.0,1.0,2.0


In [54]:
valid_prices.groupby("is_new")["owner_count"].apply(
    lambda column: column.isna().sum()
)

is_new
False      46182
True     1134962
Name: owner_count, dtype: int64

In [55]:
valid_prices.loc[
    (valid_prices["is_new"] == True) &
    (valid_prices["owner_count"] == 2)
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
22518,Warren,48093,V8,False,Gasoline,False,420.0,True,Cadillac,5.0,Escalade,2.0,86550.0,False,4.600000,A,Luxury 4WD,4WD,2019
151345,Upper Saddle River,7458,I4,False,Gasoline,False,280.0,True,Alfa Romeo,11531.0,Giulia,2.0,43385.0,False,4.625000,A,AWD,AWD,2018
261063,Newport,4953,I4,False,Gasoline,False,160.0,True,Ford,12.0,Focus,2.0,22805.0,False,NaN,A,SE Hatchback,FWD,2018
276016,North Brunswick,8902,I4,False,Gasoline,False,141.0,True,Nissan,18174.0,Rogue Sport,2.0,24999.0,False,4.485714,CVT,S AWD,AWD,2019
387620,Gurnee,60031,I4,False,Gasoline,True,173.0,True,Dodge,NaN,Journey,2.0,16995.0,False,3.920635,A,SXT FWD,FWD,2018
465929,Watertown,13601,I4,False,Gasoline,False,147.0,True,Kia,2.0,Forte,2.0,19370.0,False,5.000000,A,LX,FWD,2018
466305,Watertown,13601,V6,False,Gasoline,False,290.0,True,Kia,2.0,Sorento,2.0,42330.0,False,5.000000,A,EX V6 AWD,AWD,2019
549778,Schaumburg,60173,I4,False,Gasoline,False,141.0,True,Honda,8189.0,HR-V,2.0,26877.0,False,3.608696,CVT,EX-L AWD with Navigation,AWD,2018
732884,Newport News,23601,I4,False,Gasoline,False,245.0,True,Ford,11159.0,Escape,2.0,27495.0,False,4.125000,A,SE FWD,FWD,2018
806491,Boone,28607,V8,False,Gasoline,False,395.0,True,RAM,35895.0,1500,2.0,36186.0,False,4.482759,A,Laramie Quad Cab 4WD,4WD,2019


CHECK POINT 3 ---- NOT COMPLETE 
We need to determine whether is_new or owner_count is unreliable before cleaning either one.

CHECKPOINT 4----> Year Validation


In [56]:
valid_prices['year'].min()

np.int64(1915)

In [57]:
valid_prices.loc[
    valid_prices['year'] == 2021,
    [
        "make_name", "model_name", "price", "mileage", "is_new", "year"
    ]
   
].head(60)

,make_name,model_name,price,mileage,is_new,year
602,Jeep,Compass,26111.0,0.0,True,2021
610,Jeep,Compass,26329.0,0.0,True,2021
657,Jeep,Compass,27381.0,0.0,True,2021
668,Jeep,Compass,27593.0,0.0,True,2021
675,Jeep,Compass,28393.0,0.0,True,2021
719,Kia,Soul,19245.0,12.0,True,2021
731,Kia,Soul,19245.0,6.0,True,2021
743,Jeep,Compass,29151.0,0.0,True,2021
753,Kia,Soul,20905.0,12.0,True,2021
769,Kia,Soul,20905.0,16.0,True,2021


In [58]:
valid_prices.loc[
    valid_prices['year'] == 2021,
    [
        "make_name", "model_name", "price", "mileage", "is_new", "year"
    ]
].head(30)

,make_name,model_name,price,mileage,is_new,year
602,Jeep,Compass,26111.0,0.0,True,2021
610,Jeep,Compass,26329.0,0.0,True,2021
657,Jeep,Compass,27381.0,0.0,True,2021
668,Jeep,Compass,27593.0,0.0,True,2021
675,Jeep,Compass,28393.0,0.0,True,2021
719,Kia,Soul,19245.0,12.0,True,2021
731,Kia,Soul,19245.0,6.0,True,2021
743,Jeep,Compass,29151.0,0.0,True,2021
753,Kia,Soul,20905.0,12.0,True,2021
769,Kia,Soul,20905.0,16.0,True,2021


In [59]:
valid_prices.loc[
    valid_prices["is_new"] == True
].nlargest(20, "year")[
    ["make_name", "model_name", "year", "mileage", "price", "is_new"]
]

,make_name,model_name,year,mileage,price,is_new
602,Jeep,Compass,2021,0.0,26111.0,True
610,Jeep,Compass,2021,0.0,26329.0,True
657,Jeep,Compass,2021,0.0,27381.0,True
668,Jeep,Compass,2021,0.0,27593.0,True
675,Jeep,Compass,2021,0.0,28393.0,True
719,Kia,Soul,2021,12.0,19245.0,True
731,Kia,Soul,2021,6.0,19245.0,True
743,Jeep,Compass,2021,0.0,29151.0,True
753,Kia,Soul,2021,12.0,20905.0,True
769,Kia,Soul,2021,16.0,20905.0,True


In [60]:
valid_prices.loc[
    (valid_prices["year"] == 1915) &
    (valid_prices["is_new"] == True),
    [
        "make_name",
        "model_name",
        "price",
        "mileage",
        "is_new",
        "year",
        "owner_count"
    ]
].head(30)

,make_name,model_name,price,mileage,is_new,year,owner_count


CHECK POINT 4 ----> COMPLETE
Keep the full year range 1915–2021. No invalid years were found.

CHECK POINT 5 ----> HORSEPOWER

In [61]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663247,Vallejo,94591,V6,NaN,Gasoline,NaN,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,NaN,4.533333,A,LS FWD,FWD,2020
2663248,Napa,94559,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


In [62]:
valid_prices['horsepower'].isna().sum()

np.int64(150520)

In [63]:
valid_prices.loc[valid_prices['horsepower'].isna()].head(30)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
50,Bronx,10466,NaN,False,NaN,False,NaN,False,Subaru,19801.0,Impreza,1.0,17300.0,False,2.800000,CVT,2.0i Touring Wagon AWD,NaN,2018
162,Bronx,10466,V6,False,Gasoline,False,NaN,False,Mercedes-Benz,41672.0,C-Class,2.0,12500.0,False,2.800000,A,NaN,NaN,2012
261,Bronx,10466,I4,False,Gasoline,False,NaN,False,Volkswagen,42154.0,GTI,1.0,24300.0,False,2.800000,Dual Clutch,NaN,NaN,2018
272,Bronx,10466,NaN,False,Electric,False,NaN,False,Kia,26155.0,Soul EV,1.0,13000.0,False,2.800000,A,FWD,FWD,2017
407,Woodbury,11797,V6,False,Gasoline,False,NaN,False,Porsche,62385.0,Panamera,2.0,26995.0,False,2.963636,A,NaN,NaN,2012
421,East Hartford,6108,V6,False,Gasoline,True,NaN,False,Ford,93707.0,F-150,1.0,17433.0,False,4.377778,A,NaN,NaN,2016
440,East Hartford,6108,I4,NaN,Gasoline,NaN,NaN,True,Jeep,2.0,Compass,NaN,22256.0,NaN,4.377778,A,NaN,NaN,2020
485,Bohemia,11716,I5,False,Gasoline,True,NaN,False,Volkswagen,45063.0,Golf,1.0,10646.0,False,3.647059,A,NaN,NaN,2013
498,Bronx,10466,V6,False,Gasoline,False,NaN,False,Lexus,31160.0,GS 350,1.0,30000.0,False,2.800000,A,NaN,NaN,2016
499,Bohemia,11716,V6,False,Gasoline,False,NaN,False,Mercedes-Benz,38885.0,E-Class,1.0,21561.0,False,3.647059,A,NaN,NaN,2014


In [64]:
valid_prices.loc[valid_prices['horsepower'] == 0]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year


In [65]:
valid_prices.loc[valid_prices['horsepower'] < 0]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year


In [66]:
valid_prices.loc[
    valid_prices["horsepower"] == valid_prices["horsepower"].max()
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
810693,Charlotte,28203,W16,False,Gasoline,False,1001.0,False,Bugatti,8597.0,Veyron,9.0,1244996.0,False,4.6875,A,16.4 Coupe AWD,AWD,2008
2545871,Costa Mesa,92627,W16,False,Gasoline,False,1001.0,False,Bugatti,9791.0,Veyron,4.0,955000.0,False,5.0000,A,16.4 Coupe AWD,AWD,2008


In [67]:
valid_prices['horsepower'].min()

np.float64(55.0)

In [68]:
valid_prices.loc[
    valid_prices["horsepower"] == valid_prices["horsepower"].min()
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
1633111,Brookville,45309,I3,False,Gasoline,False,55.0,False,Chevrolet,147375.0,Metro,1.0,1000.0,False,3.857143,M,Hatchback FWD,FWD,2000


CHECKPOINT ----> 5 COMPLETE


Horsepower validation conclusion

No zero or negative values

Minimum and maximum are plausible

Missing values exist, but their cause is unknown

Leave them as NaN until the missing-value phase

CHECK POINT 6 -----> Seller rating validation


In [69]:
valid_prices['seller_rating'].min()

np.float64(1.0)

In [70]:
valid_prices['seller_rating'].max()

np.float64(5.0)

In [71]:
valid_prices['seller_rating'].isna().sum()

np.int64(38290)

In [72]:
valid_prices.loc[valid_prices['seller_rating'].isna()].head(30)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
10,Guaynabo,969,I4,False,Gasoline,False,237.0,False,Alfa Romeo,301.0,4C,2.0,97579.0,False,NaN,A,Launch Edition Coupe RWD,RWD,2015
12,Guaynabo,969,I6,False,Gasoline,False,320.0,False,BMW,6903.0,3 Series,2.0,58995.0,False,NaN,A,340i xDrive Sedan AWD,AWD,2016
17385,Auburn Hills,48326,I4,NaN,Gasoline,NaN,120.0,True,Kia,34.0,Rio,NaN,16651.0,NaN,NaN,CVT,LX FWD,FWD,2020
17400,Auburn Hills,48326,V6 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,283.0,False,Dodge,38358.0,Grand Caravan,1.0,18877.0,False,NaN,A,SXT FWD,FWD,2019
17407,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,17.0,Forte,NaN,20204.0,NaN,NaN,CVT,LXS FWD,FWD,2020
17413,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,14.0,Soul,NaN,19888.0,NaN,NaN,CVT,LX FWD,FWD,2021
17419,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,12.0,Soul,NaN,19888.0,NaN,NaN,CVT,LX FWD,FWD,2021
17420,Auburn Hills,48326,I4,NaN,Gasoline,NaN,147.0,True,Kia,19.0,Forte,NaN,20186.0,NaN,NaN,CVT,LXS FWD,FWD,2020
17529,Auburn Hills,48326,I4,NaN,Gasoline,NaN,120.0,True,Kia,26.0,Rio,NaN,16651.0,NaN,NaN,CVT,LX FWD,FWD,2020


CHECKPOINT 6 ---> COMPLETE

MISSING_VALUE TREATMENT
Start off with Frame_damaged column

In [73]:
valid_prices.loc[valid_prices['frame_damaged']== True]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
383,Linden,7036,I4,True,Gasoline,True,140.0,False,Honda,140000.0,Civic Coupe,4.0,4499.0,False,4.666667,A,EX,FWD,2007
393,Linden,7036,V6,True,Gasoline,False,253.0,False,Acura,145339.0,MDX,3.0,4999.0,False,4.666667,A,"AWD with Touring Package, Navigation, and Ente...",AWD,2006
395,Linden,7036,I4,True,Gasoline,True,160.0,False,Acura,159291.0,RSX,3.0,4999.0,False,4.666667,M,FWD with Leather,FWD,2002
404,Linden,7036,V6,True,Gasoline,False,228.0,False,Mercedes-Benz,91899.0,C-Class,5.0,6999.0,False,4.666667,A,C 280 4MATIC Luxury AWD,AWD,2007
406,Woodbury,11797,I3,True,Gasoline,True,357.0,False,BMW,63289.0,i8,4.0,48995.0,False,2.963636,A,Coupe AWD,AWD,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2659848,Vallejo,94589,I4,True,Gasoline,False,190.0,False,Honda,137518.0,Accord Coupe,2.0,7800.0,False,4.625000,A,EX,FWD,2009
2661247,Napa,94559,V6,True,Gasoline,False,300.0,False,Nissan,36548.0,Maxima,1.0,21991.0,False,4.285714,CVT,SL FWD,FWD,2018
2661428,Lakeport,95453,I4,True,Gasoline,False,149.0,False,Chevrolet,26322.0,Volt,1.0,15649.0,False,4.538462,A,LT FWD,FWD,2017
2662530,Ukiah,95482,I4,True,Gasoline,True,185.0,False,Honda,91416.0,Accord Coupe,2.0,13995.0,False,5.000000,CVT,LX-S,FWD,2014


In [74]:
valid_prices.loc[valid_prices['frame_damaged']== False]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
5,San Juan,922,I4,False,Gasoline,False,247.0,True,Land Rover,12.0,Range Rover Velar,NaN,66903.0,False,3.000000,A,P250 R-Dynamic S AWD,AWD,2020
9,San Juan,922,I4,False,Gasoline,False,296.0,False,Land Rover,254.0,Range Rover Evoque,NaN,84399.0,False,3.000000,A,P300 R-Dynamic SE AWD,AWD,2020
10,Guaynabo,969,I4,False,Gasoline,False,237.0,False,Alfa Romeo,301.0,4C,2.0,97579.0,False,NaN,A,Launch Edition Coupe RWD,RWD,2015
12,Guaynabo,969,I6,False,Gasoline,False,320.0,False,BMW,6903.0,3 Series,2.0,58995.0,False,NaN,A,340i xDrive Sedan AWD,AWD,2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663245,Ukiah,95482,V6,False,Gasoline,False,278.0,False,Toyota,20009.0,Tacoma,1.0,40993.0,False,5.000000,A,TRD Sport V6 Double Cab 4WD,4WD,2017
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663248,Napa,94559,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


In [75]:
valid_prices.loc[valid_prices['frame_damaged'].isna()]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
6,Bayamon,960,I4,NaN,Gasoline,NaN,186.0,True,Mazda,14.0,MAZDA3,NaN,23695.0,NaN,2.800000,A,Sedan FWD,FWD,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663231,Napa,94559,I4,NaN,Gasoline,NaN,149.0,True,Nissan,0.0,Sentra,NaN,22578.0,NaN,4.333333,CVT,SV FWD,FWD,2020
2663234,Napa,94559,I4,NaN,Gasoline,NaN,141.0,True,Nissan,0.0,Rogue Sport,NaN,22667.0,NaN,4.333333,CVT,SV FWD,FWD,2020
2663236,Ukiah,95482,V6,NaN,Gasoline,NaN,375.0,True,Ford,NaN,Expedition,NaN,72635.0,NaN,5.000000,A,Limited MAX 4WD,4WD,2020
2663243,Napa,94559,I4,NaN,Gasoline,NaN,131.0,True,Nissan,0.0,NV200,NaN,22901.0,NaN,4.333333,CVT,S FWD,FWD,2020


In [76]:
valid_prices.groupby("is_new")["frame_damaged"].value_counts(dropna=False)

is_new  frame_damaged
False   False            1509931
        True               14945
        NaN                 1954
True    NaN              1094222
        False              41705
Name: count, dtype: int64

In [77]:
valid_prices.loc[valid_prices['has_accidents']== True].tail(30)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
2662662,Fort Bragg,95437,I4,False,Gasoline,True,148.0,False,Mitsubishi,44918.0,Lancer,1.0,10995.0,False,5.000000,CVT,ES,FWD,2017
2662670,Fort Bragg,95437,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,True,342.0,False,Chevrolet,94000.0,Express,2.0,13995.0,False,5.000000,A,3500 1LT Extended RWD,RWD,2016
2662705,Fort Bragg,95437,I4,False,Gasoline,True,220.0,False,Audi,73380.0,Q5,2.0,18995.0,False,5.000000,A,2.0T quattro Premium Plus AWD,AWD,2015
2662722,Fort Bragg,95437,I4,False,Gasoline,True,158.0,False,Honda,13750.0,Civic,1.0,18995.0,False,5.000000,CVT,LX FWD,FWD,2019
2662771,Fort Bragg,95437,I4 Hybrid,False,Hybrid,True,134.0,False,Toyota,85405.0,Prius,4.0,13995.0,False,5.000000,CVT,One,FWD,2015
2662774,Santa Rosa,95407,V6,False,Gasoline,True,302.0,False,Mercedes-Benz,72887.0,C-Class,2.0,14900.0,False,4.200000,A,C 350 Sport,RWD,2013
2662787,Fairfield,94533,I4,False,Gasoline,True,204.0,False,Audi,3737.0,A3 Sportback,1.0,28998.0,False,4.272727,A,e-tron 1.4T Prestige FWD,FWD,2018
2662803,Fairfield,94533,I4 Hybrid,False,Hybrid,True,212.0,False,Honda,28887.0,Accord Hybrid,1.0,26998.0,False,4.272727,A,Touring,FWD,2018
2662817,Santa Rosa,95407,I4 Hybrid,False,Hybrid,True,110.0,False,Toyota,98644.0,Prius,1.0,7100.0,False,4.200000,CVT,FWD,FWD,2006
2662834,Ukiah,95482,I4,False,Gasoline,True,245.0,False,Ford,37748.0,Escape,1.0,18870.0,False,5.000000,A,SE FWD,FWD,2019


In [78]:
valid_prices.loc[valid_prices['has_accidents']== False]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
5,San Juan,922,I4,False,Gasoline,False,247.0,True,Land Rover,12.0,Range Rover Velar,NaN,66903.0,False,3.000000,A,P250 R-Dynamic S AWD,AWD,2020
9,San Juan,922,I4,False,Gasoline,False,296.0,False,Land Rover,254.0,Range Rover Evoque,NaN,84399.0,False,3.000000,A,P300 R-Dynamic SE AWD,AWD,2020
10,Guaynabo,969,I4,False,Gasoline,False,237.0,False,Alfa Romeo,301.0,4C,2.0,97579.0,False,NaN,A,Launch Edition Coupe RWD,RWD,2015
12,Guaynabo,969,I6,False,Gasoline,False,320.0,False,BMW,6903.0,3 Series,2.0,58995.0,False,NaN,A,340i xDrive Sedan AWD,AWD,2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663244,Fremont,94538,I4 Hybrid,False,Hybrid,False,110.0,False,Toyota,151340.0,Prius,1.0,5371.0,False,5.000000,CVT,FWD,FWD,2006
2663245,Ukiah,95482,V6,False,Gasoline,False,278.0,False,Toyota,20009.0,Tacoma,1.0,40993.0,False,5.000000,A,TRD Sport V6 Double Cab 4WD,4WD,2017
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


In [79]:
valid_prices.loc[valid_prices['has_accidents'].isna()]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,NaN,Gasoline,NaN,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,NaN,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,NaN,3.000000,A,S AWD,AWD,2020
3,San Juan,922,V6,NaN,Gasoline,NaN,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,NaN,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,NaN,Gasoline,NaN,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,NaN,3.000000,A,S AWD,AWD,2020
6,Bayamon,960,I4,NaN,Gasoline,NaN,186.0,True,Mazda,14.0,MAZDA3,NaN,23695.0,NaN,2.800000,A,Sedan FWD,FWD,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663231,Napa,94559,I4,NaN,Gasoline,NaN,149.0,True,Nissan,0.0,Sentra,NaN,22578.0,NaN,4.333333,CVT,SV FWD,FWD,2020
2663234,Napa,94559,I4,NaN,Gasoline,NaN,141.0,True,Nissan,0.0,Rogue Sport,NaN,22667.0,NaN,4.333333,CVT,SV FWD,FWD,2020
2663236,Ukiah,95482,V6,NaN,Gasoline,NaN,375.0,True,Ford,NaN,Expedition,NaN,72635.0,NaN,5.000000,A,Limited MAX 4WD,4WD,2020
2663243,Napa,94559,I4,NaN,Gasoline,NaN,131.0,True,Nissan,0.0,NV200,NaN,22901.0,NaN,4.333333,CVT,S FWD,FWD,2020


In [80]:
valid_prices.groupby("mileage")["has_accidents"].value_counts(dropna=False)

mileage   has_accidents
0.0       NaN              173858
          False              5102
          True                  9
1.0       NaN               51003
          False              1600
                            ...  
398823.0  True                  1
399496.0  True                  1
399578.0  False                 1
399900.0  False                 1
400000.0  True                  1
Name: count, Length: 321304, dtype: int64

In [81]:
valid_prices.groupby("is_new")["salvage"].value_counts(dropna=False)

is_new  salvage
False   False      1514420
        True         10456
        NaN           1954
True    NaN        1094222
        False        41701
        True             4
Name: count, dtype: int64

In [82]:
damage_columns = ["frame_damaged", "has_accidents", "salvage"]

valid_prices[damage_columns] = valid_prices[damage_columns].fillna("Not Reported")

In [83]:
valid_prices[damage_columns].isna().sum()

frame_damaged    0
has_accidents    0
salvage          0
dtype: int64

In [84]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,Not Reported,Gasoline,Not Reported,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,Not Reported,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,Not Reported,3.000000,A,S AWD,AWD,2020
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,NaN,M,Base,AWD,2016
3,San Juan,922,V6,Not Reported,Gasoline,Not Reported,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,Not Reported,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,Not Reported,3.000000,A,S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,4.272727,A,1.5T LT FWD,FWD,2018
2663247,Vallejo,94591,V6,Not Reported,Gasoline,Not Reported,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,Not Reported,4.533333,A,LS FWD,FWD,2020
2663248,Napa,94559,NaN,False,NaN,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,4.142857,A,SE,FWD,2016
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,4.272727,A,20d Premium AWD,AWD,2017


In [85]:
valid_prices.groupby("is_new")["owner_count"].apply(lambda s: s.isna().sum())

is_new
False      46182
True     1134962
Name: owner_count, dtype: int64

In [86]:
valid_prices.loc[valid_prices["is_new"] == True]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,seller_rating,transmission,trim_name,wheel_system,year
0,Bayamon,960,I4,Not Reported,Gasoline,Not Reported,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,Not Reported,2.800000,A,Latitude FWD,FWD,2019
1,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,Not Reported,3.000000,A,S AWD,AWD,2020
3,San Juan,922,V6,Not Reported,Gasoline,Not Reported,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,Not Reported,3.000000,A,V6 HSE AWD,AWD,2020
4,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,Not Reported,3.000000,A,S AWD,AWD,2020
5,San Juan,922,I4,False,Gasoline,False,247.0,True,Land Rover,12.0,Range Rover Velar,NaN,66903.0,False,3.000000,A,P250 R-Dynamic S AWD,AWD,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663231,Napa,94559,I4,Not Reported,Gasoline,Not Reported,149.0,True,Nissan,0.0,Sentra,NaN,22578.0,Not Reported,4.333333,CVT,SV FWD,FWD,2020
2663234,Napa,94559,I4,Not Reported,Gasoline,Not Reported,141.0,True,Nissan,0.0,Rogue Sport,NaN,22667.0,Not Reported,4.333333,CVT,SV FWD,FWD,2020
2663236,Ukiah,95482,V6,Not Reported,Gasoline,Not Reported,375.0,True,Ford,NaN,Expedition,NaN,72635.0,Not Reported,5.000000,A,Limited MAX 4WD,4WD,2020
2663243,Napa,94559,I4,Not Reported,Gasoline,Not Reported,131.0,True,Nissan,0.0,NV200,NaN,22901.0,Not Reported,4.333333,CVT,S FWD,FWD,2020


In [87]:
valid_prices["is_new"].value_counts()

is_new
False    1526830
True     1135927
Name: count, dtype: int64

In [88]:
1134962 / (valid_prices['is_new']== True).sum()*100

np.float64(99.91504735779677)

In [89]:
46,182 / (valid_prices['is_new']== False).sum()*100

(46, np.float64(0.011920122083008586))

In [90]:
valid_prices.groupby("is_new")["owner_count"].max()

is_new
False    19.0
True      2.0
Name: owner_count, dtype: float64

In [91]:
used_vehicles = valid_prices.loc[
    valid_prices["is_new"] == False
].copy()

In [92]:
used_vehicles.assign(
    owner_count_missing=used_vehicles["owner_count"].isna()
).groupby("owner_count_missing")[
    ["year", "mileage", "price"]
].median()

,year,mileage,price
owner_count_missing,,,
False,2017.0,41693.0,18998.0
True,2019.0,16600.0,32991.0


In [93]:
valid_prices["owner_count_missing"] = (
    valid_prices["owner_count"].isna().astype(int)
)

In [94]:
valid_prices["owner_count_missing"].value_counts()

owner_count_missing
0    1481613
1    1181144
Name: count, dtype: int64

In [95]:
valid_prices.groupby("is_new")["mileage"].apply(
    lambda column: column.isna().sum()
)

is_new
False    14912
True     87606
Name: mileage, dtype: int64

In [96]:
14912 / (valid_prices['is_new']== False).sum()*100

np.float64(0.9766640686913408)

In [97]:
87606 / (valid_prices['is_new']== True).sum()*100

np.float64(7.71229137083633)

In [98]:
valid_prices["mileage_missing"] = (
    valid_prices["mileage"].isna().astype(int)
)

In [99]:
valid_prices["mileage_missing"].value_counts()


mileage_missing
0    2560239
1     102518
Name: count, dtype: int64

In [100]:
valid_prices.groupby("is_new")["horsepower"].apply(
    lambda column: column.isna().sum()
)

is_new
False    61012
True     89508
Name: horsepower, dtype: int64

In [101]:
89508 / (valid_prices['is_new']== True).sum()*100

np.float64(7.879731708111525)

In [102]:
61012 / (valid_prices['is_new']== False).sum()*100

np.float64(3.9959916952116474)

In [103]:
valid_prices["horsepower_missing"] = (
    valid_prices["horsepower"].isna().astype(int)
)
valid_prices["horsepower_missing"].value_counts()

horsepower_missing
0    2512237
1     150520
Name: count, dtype: int64

In [104]:
valid_prices.loc[
    valid_prices["seller_rating"].isna(),
    "dealer_zip"
].value_counts(dropna=False).head(20)


dealer_zip
92504    396
33157    394
33619    318
68154    297
75067    273
28546    269
46792    269
84401    258
65775    246
33411    232
90007    218
95340    216
79936    212
66062    209
95991    207
28345    206
84042    198
5843     191
87402    191
68147    185
Name: count, dtype: int64

In [105]:
seller_rating_by_zip = (
    valid_prices.groupby("dealer_zip", dropna=False)["seller_rating"]
    .agg(
        total_listings="size",
        missing_ratings=lambda column: column.isna().sum()
    )
)

In [106]:
seller_rating_by_zip["missing_percentage"] = (
    seller_rating_by_zip["missing_ratings"]
    / seller_rating_by_zip["total_listings"]
    * 100
)

In [107]:
seller_rating_by_zip.loc[
    seller_rating_by_zip["total_listings"] >= 100
].sort_values(
    "missing_percentage",
    ascending=False
).head(20)

,total_listings,missing_ratings,missing_percentage
dealer_zip,,,
48170,101,101,100.000000
13669,130,130,100.000000
13838,147,147,100.000000
5843,191,191,100.000000
70563,134,134,100.000000
72315,100,100,100.000000
59711,102,102,100.000000
84078,168,168,100.000000
80203,161,161,100.000000


In [108]:
valid_prices.drop(columns=["seller_rating"], inplace=True)

In [109]:
valid_prices.isna().sum().sort_values(ascending=False)

owner_count            1181144
horsepower              150520
wheel_system            127983
trim_name               102958
mileage                 102518
engine_type              87885
fuel_type                72606
transmission             54944
city                         0
frame_damaged                0
dealer_zip                   0
make_name                    0
has_accidents                0
price                        0
model_name                   0
is_new                       0
salvage                      0
year                         0
owner_count_missing          0
mileage_missing              0
horsepower_missing           0
dtype: int64

In [110]:
valid_prices.groupby("is_new")["wheel_system"].apply(
    lambda column: column.isna().sum()
)

is_new
False    48220
True     79763
Name: wheel_system, dtype: int64

In [111]:
79763 / (valid_prices['is_new']== True).sum()*100

np.float64(7.021842072597975)

In [112]:
48220 / (valid_prices['is_new']== False).sum()*100

np.float64(3.158177400234473)

In [113]:
valid_prices["wheel_system"].value_counts(dropna=False)

wheel_system
FWD    1082273
AWD     622019
4WD     541022
RWD     180847
NaN     127983
4X2     108613
Name: count, dtype: int64

In [114]:
valid_prices.loc[
    valid_prices['wheel_system'].isna(),
    ['trim_name', 'wheel_system']

].head(50)

,trim_name,wheel_system
50,2.0i Touring Wagon AWD,NaN
162,NaN,NaN
261,NaN,NaN
407,NaN,NaN
421,NaN,NaN
440,NaN,NaN
485,NaN,NaN
498,NaN,NaN
499,NaN,NaN
594,NaN,NaN


In [115]:
valid_prices.loc[
    valid_prices['wheel_system'].isna(),
    'trim_name'
].value_counts(dropna=False).head(30)

trim_name
NaN                                 102490
SEL FWD                               2898
LX FWD                                1595
Altitude RWD                          1438
Limited RWD                           1425
SEL Plus FWD                          1375
Ultimate AWD                          1354
1.4T Comfortline FWD                  1066
Laredo 4WD                             947
Laramie Crew Cab LWB DRW 4WD           767
Laredo E 4WD                           765
Limited FWD                            748
Limited AWD                            709
SE AWD                                 707
SE FWD                                 549
Limited AWD with Captains Chairs       406
SEL Plus AWD                           342
GT Line FWD                            274
Limited X RWD                          252
SE I4 Sedan                            225
Limited X 4WD                          220
DRW 4WD                                203
2.0i Convenience Sedan AWD             191
P

In [116]:
valid_prices.loc[
    valid_prices["trim_name"].str.contains(
        r"\bAWD\b",
        case=False,
        na=False
    )
    & valid_prices["wheel_system"].notna()
    & (valid_prices["wheel_system"] != "AWD"),
    [
        "make_name",
        "model_name",
        "trim_name",
        "wheel_system"
    ]
].head(30)

,make_name,model_name,trim_name,wheel_system
72,Chevrolet,Equinox,1.5T LT AWD,4WD
100,Chevrolet,Equinox,1.5T LT AWD,4WD
402,Chevrolet,Traverse,LT Cloth AWD,4WD
409,Cadillac,XT5,Luxury AWD,4WD
426,Chevrolet,Equinox,1.5T Premier AWD,4WD
436,Chevrolet,Traverse,Premier AWD,4WD
473,Chevrolet,Traverse,LT Cloth AWD,4WD
483,Chevrolet,Equinox,1.5T Premier AWD,4WD
511,Chevrolet,Equinox,1.5T LT AWD,4WD
538,Chevrolet,Traverse,LT Cloth AWD,4WD


In [117]:
valid_prices.loc[
    valid_prices["trim_name"].str.contains(
        r"\bAWD\b",
        case=False,
        na=False
    ),
    "wheel_system"
].value_counts(dropna=False)

wheel_system
AWD    555168
4WD     52099
NaN      5266
FWD        37
RWD         5
Name: count, dtype: int64

In [118]:
group_consistency = (
    valid_prices
    .dropna(subset=["trim_name", "wheel_system"])
    .groupby(["make_name", "model_name", "trim_name", "year"])["wheel_system"]
    .agg(
        recorded_rows="size",
        unique_wheel_systems="nunique"
    )
)

In [119]:
group_consistency["unique_wheel_systems"].value_counts()

unique_wheel_systems
1    40063
2       20
Name: count, dtype: int64

In [120]:
group_keys = ["make_name", "model_name", "trim_name", "year"]

consistent_groups = (
    group_consistency
    .loc[group_consistency["unique_wheel_systems"] == 1]
    .reset_index()[group_keys]
)

missing_wheel_system = valid_prices.loc[
    valid_prices["wheel_system"].isna()
]

recoverable_rows = missing_wheel_system.merge(
    consistent_groups,
    on=group_keys,
    how="inner"
)

recoverable_rows.shape[0]

13242

In [121]:
recoverable_rows.shape[0] / valid_prices["wheel_system"].isna().sum() * 100


np.float64(10.34668666932327)

In [122]:
recoverable_rows


,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
0,East Hartford,6108,V6,Not Reported,Gasoline,Not Reported,NaN,True,Jeep,1.0,Grand Cherokee,NaN,30777.0,Not Reported,A,Laredo E 4WD,NaN,2020,1,0,1
1,East Hartford,6108,V6,Not Reported,Gasoline,Not Reported,NaN,True,Jeep,15.0,Grand Cherokee,NaN,30948.0,Not Reported,A,Laredo E 4WD,NaN,2020,1,0,1
2,East Hartford,6108,V6,Not Reported,Gasoline,Not Reported,NaN,True,Jeep,1.0,Grand Cherokee,NaN,30991.0,Not Reported,A,Laredo E 4WD,NaN,2020,1,0,1
3,West Nyack,10994,I4,Not Reported,Gasoline,Not Reported,NaN,True,Hyundai,10.0,Tucson,NaN,24606.0,Not Reported,A,SE AWD,NaN,2021,1,0,1
4,West Nyack,10994,I4,Not Reported,Gasoline,Not Reported,NaN,True,Hyundai,5.0,Tucson,NaN,24606.0,Not Reported,A,SE AWD,NaN,2021,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13237,Santa Rosa,95407,NaN,Not Reported,Electric,Not Reported,NaN,True,Hyundai,14.0,Kona Electric,NaN,43505.0,Not Reported,A,Limited FWD,NaN,2020,1,0,1
13238,Vacaville,95687,I4,Not Reported,Gasoline,Not Reported,NaN,True,Hyundai,7.0,Ioniq Hybrid Plug-In,NaN,27320.0,Not Reported,A,SE FWD,NaN,2020,1,0,1
13239,Santa Rosa,95407,I4,Not Reported,Gasoline,Not Reported,NaN,True,Hyundai,12.0,Kona,NaN,30845.0,Not Reported,A,Ultimate AWD,NaN,2020,1,0,1
13240,Santa Rosa,95407,V6,Not Reported,Gasoline,Not Reported,NaN,True,Jeep,10.0,Grand Cherokee,NaN,47885.0,Not Reported,A,Limited X 4WD,NaN,2020,1,0,1


In [123]:
known_wheel_systems = valid_prices.loc[
    valid_prices['wheel_system'].notna(),
    ['make_name', 'model_name', 'trim_name', 'year', 'wheel_system']
]

In [124]:
known_wheel_systems.head(30)

,make_name,model_name,trim_name,year,wheel_system
0,Jeep,Renegade,Latitude FWD,2019,FWD
1,Land Rover,Discovery Sport,S AWD,2020,AWD
2,Subaru,WRX STI,Base,2016,AWD
3,Land Rover,Discovery,V6 HSE AWD,2020,AWD
4,Land Rover,Discovery Sport,S AWD,2020,AWD
5,Land Rover,Range Rover Velar,P250 R-Dynamic S AWD,2020,AWD
6,Mazda,MAZDA3,Sedan FWD,2019,FWD
7,Land Rover,Range Rover Velar,P250 R-Dynamic S AWD,2020,AWD
8,Land Rover,Discovery Sport,S AWD,2020,AWD
9,Land Rover,Range Rover Evoque,P300 R-Dynamic SE AWD,2020,AWD


In [125]:
known_wheel_systems.shape

(2534774, 5)

In [126]:
known_wheel_systems.describe

<bound method NDFrame.describe of           make_name       model_name        trim_name  year wheel_system
0              Jeep         Renegade     Latitude FWD  2019          FWD
1        Land Rover  Discovery Sport            S AWD  2020          AWD
2            Subaru          WRX STI             Base  2016          AWD
3        Land Rover        Discovery       V6 HSE AWD  2020          AWD
4        Land Rover  Discovery Sport            S AWD  2020          AWD
...             ...              ...              ...   ...          ...
2663246   Chevrolet          Equinox      1.5T LT FWD  2018          FWD
2663247   Chevrolet         Traverse           LS FWD  2020          FWD
2663248        Ford           Fusion               SE  2016          FWD
2663249      Jaguar               XE  20d Premium AWD  2017          AWD
2663250      Nissan            Rogue    2017.5 SV FWD  2017          FWD

[2534774 rows x 5 columns]>

A configuration-based drivetrain recovery method was identified as a possible
future improvement. The baseline version uses an `Unknown` category to keep
the preprocessing pipeline simple and avoid unsupported inference.

In [127]:
valid_prices.loc[
    valid_prices["wheel_system"].isna(),
    "wheel_system"
] = "Unknown"

In [128]:
valid_prices["wheel_system"].isna().sum()

np.int64(0)

In [129]:
valid_prices.groupby("is_new")["trim_name"].apply(
    lambda column: column.isna().sum()
)

is_new
False    41971
True     60987
Name: trim_name, dtype: int64

In [130]:
60987 / (valid_prices['is_new']== True).sum()*100

np.float64(5.368918953418662)

In [131]:
41971 / (valid_prices['is_new']== False).sum()*100

np.float64(2.748898043659085)

In [132]:
valid_prices['trim_name'].isna().sum()

np.int64(102958)

In [133]:
valid_prices.head(5)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
0,Bayamon,960,I4,Not Reported,Gasoline,Not Reported,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,Not Reported,A,Latitude FWD,FWD,2019,1,0,0
1,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,Not Reported,A,S AWD,AWD,2020,1,0,0
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,M,Base,AWD,2016,0,1,0
3,San Juan,922,V6,Not Reported,Gasoline,Not Reported,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,Not Reported,A,V6 HSE AWD,AWD,2020,1,0,0
4,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,Not Reported,A,S AWD,AWD,2020,1,0,0


In [134]:
valid_prices['trim_name'] = valid_prices['trim_name'].fillna('Unknown')

In [135]:
valid_prices["trim_name"].isna().sum()

np.int64(0)

In [136]:
engine_missing = (
    valid_prices.groupby("is_new")["engine_type"]
    .apply(lambda column: column.isna().sum())
)

group_totals = valid_prices["is_new"].value_counts()

engine_missing_percentage = (
    engine_missing / group_totals * 100
)

engine_missing_percentage

is_new
False    2.537938
True     4.325542
dtype: float64

In [137]:
valid_prices.loc[
    valid_prices["engine_type"].isna(),
    "fuel_type"
].value_counts(dropna=False)

fuel_type
NaN         72606
Electric    14072
Diesel        759
Gasoline      448
Name: count, dtype: int64

In [138]:
electric_engine_missing = (
    valid_prices["engine_type"].isna() &
    valid_prices["fuel_type"].eq("Electric")
)

valid_prices.loc[
    electric_engine_missing,
    "engine_type"
] = "Electric Motor"

valid_prices["engine_type"] = (
    valid_prices["engine_type"].fillna("Unknown")
)

In [139]:
valid_prices["engine_type"].isna().sum()

np.int64(0)

In [140]:
valid_prices["fuel_type"] = valid_prices["fuel_type"].fillna("Unknown")

In [141]:
valid_prices["fuel_type"].isna().sum()

np.int64(0)

In [142]:
valid_prices["transmission"].value_counts(dropna=False)

transmission
A              2192909
CVT             357774
NaN              54944
M                47216
Dual Clutch       9914
Name: count, dtype: int64

In [143]:
valid_prices.groupby("is_new")["transmission"].apply(
    lambda column: column.isna().sum()
)

is_new
False    25895
True     29049
Name: transmission, dtype: int64

In [144]:
25895 / (valid_prices['is_new']== False).sum()*100

np.float64(1.6959975897775128)

In [145]:
29049 / (valid_prices['is_new']== True).sum()*100

np.float64(2.5572946148828226)

In [146]:
valid_prices.loc[
    valid_prices["transmission"].isna(),
    "trim_name"
].notna().sum()

np.int64(54944)

In [147]:
valid_prices.loc[
    valid_prices["transmission"].isna() &
    (valid_prices["trim_name"] != "Unknown"),
    "trim_name"
].shape[0]

47976

In [148]:
transmission_terms = r"\b(?:CVT|Manual|Automatic|Dual Clutch)\b"

explicit_transmission_mask = (
    valid_prices["transmission"].isna()
    & (valid_prices["trim_name"] != "Unknown")
    & valid_prices["trim_name"].str.contains(
        transmission_terms,
        case=False,
        na=False
    )
)

explicit_transmission_mask.sum()




np.int64(3)

In [149]:
valid_prices["transmission"] = (
    valid_prices["transmission"].fillna("Unknown")
)

In [150]:
valid_prices["transmission"].isna().sum()

np.int64(0)

In [151]:
valid_prices.isna().sum().sort_values(ascending=False)

owner_count            1181144
horsepower              150520
mileage                 102518
dealer_zip                   0
city                         0
fuel_type                    0
frame_damaged                0
engine_type                  0
has_accidents                0
make_name                    0
is_new                       0
model_name                   0
price                        0
salvage                      0
transmission                 0
trim_name                    0
wheel_system                 0
year                         0
owner_count_missing          0
mileage_missing              0
horsepower_missing           0
dtype: int64

## Missing-Value Treatment

The missing-value phase focused on preserving useful information without making unsupported assumptions.

### Decisions Applied

- `frame_damaged`, `has_accidents`, and `salvage`
  - Missing values were replaced with `"Not Reported"`.
  - Missingness was strongly concentrated among new vehicles, so it was not treated as `False`.

- `owner_count`
  - Missing values were left as `NaN`.
  - Added `owner_count_missing` to preserve whether the original value was unavailable.
  - Imputation will be performed later using training data only.

- `mileage`
  - Invalid mileage values were changed to `NaN`.
  - Added `mileage_missing`.
  - Missing mileage will later be estimated using vehicle year and `is_new`.

- `horsepower`
  - Missing values were left as `NaN`.
  - Added `horsepower_missing`.
  - Future imputation may use make, model, year, engine type, and trim.

- `wheel_system`, `trim_name`, `fuel_type`, and `transmission`
  - Missing values were replaced with `"Unknown"`.

- `engine_type`
  - Missing electric-vehicle engine types were labelled `"Electric Motor"`.
  - Remaining missing values were labelled `"Unknown"`.

- `seller_rating`
  - Removed because it describes seller behaviour and may not be available when the platform makes predictions.

### Future Improvements

After the baseline model is evaluated, revisit:

- configuration-based recovery of missing `wheel_system`;
- more precise mileage and horsepower imputation;
- recovery of missing engine, fuel, trim, or transmission values;
- whether missing-value indicators improve prediction accuracy.

All numerical imputation statistics must be learned from the training data only to avoid data leakage.

In [152]:
valid_prices

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
0,Bayamon,960,I4,Not Reported,Gasoline,Not Reported,177.0,True,Jeep,7.0,Renegade,NaN,23141.0,Not Reported,A,Latitude FWD,FWD,2019,1,0,0
1,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,8.0,Discovery Sport,NaN,46500.0,Not Reported,A,S AWD,AWD,2020,1,0,0
2,Guaynabo,969,H4,False,Gasoline,False,305.0,False,Subaru,NaN,WRX STI,3.0,46995.0,False,M,Base,AWD,2016,0,1,0
3,San Juan,922,V6,Not Reported,Gasoline,Not Reported,340.0,True,Land Rover,11.0,Discovery,NaN,67430.0,Not Reported,A,V6 HSE AWD,AWD,2020,1,0,0
4,San Juan,922,I4,Not Reported,Gasoline,Not Reported,246.0,True,Land Rover,7.0,Discovery Sport,NaN,48880.0,Not Reported,A,S AWD,AWD,2020,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2663246,Fairfield,94533,I4,False,Gasoline,False,170.0,False,Chevrolet,41897.0,Equinox,1.0,17998.0,False,A,1.5T LT FWD,FWD,2018,0,0,0
2663247,Vallejo,94591,V6,Not Reported,Gasoline,Not Reported,310.0,True,Chevrolet,5.0,Traverse,NaN,36490.0,Not Reported,A,LS FWD,FWD,2020,1,0,0
2663248,Napa,94559,Unknown,False,Unknown,True,240.0,False,Ford,57992.0,Fusion,2.0,12990.0,False,A,SE,FWD,2016,0,0,0
2663249,Fairfield,94533,I4 Diesel,False,Diesel,False,180.0,False,Jaguar,27857.0,XE,1.0,26998.0,False,A,20d Premium AWD,AWD,2017,0,0,0


In [153]:
valid_prices.info()

<class 'pandas.DataFrame'>
Index: 2662757 entries, 0 to 2663250
Data columns (total 21 columns):
 #   Column               Dtype  
---  ------               -----  
 0   city                 str    
 1   dealer_zip           object 
 2   engine_type          str    
 3   frame_damaged        object 
 4   fuel_type            str    
 5   has_accidents        object 
 6   horsepower           float64
 7   is_new               bool   
 8   make_name            str    
 9   mileage              float64
 10  model_name           str    
 11  owner_count          float64
 12  price                float64
 13  salvage              object 
 14  transmission         str    
 15  trim_name            str    
 16  wheel_system         str    
 17  year                 int64  
 18  owner_count_missing  int64  
 19  mileage_missing      int64  
 20  horsepower_missing   int64  
dtypes: bool(1), float64(4), int64(4), object(4), str(8)
memory usage: 556.7+ MB


In [154]:
numerical_cols = valid_prices.select_dtypes(include="number").columns.tolist()
categorical_cols = valid_prices.select_dtypes(include="object").columns.tolist()
boolean_cols = valid_prices.select_dtypes(include="bool").columns.tolist()

C:\Users\casey\AppData\Local\Temp\ipykernel_3260\1779595962.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = valid_prices.select_dtypes(include="object").columns.tolist()


In [155]:
numerical_cols.remove("price")  # target, not a feature

missing_indicators = [
    "owner_count_missing",
    "mileage_missing",
    "horsepower_missing"
]

numerical_cols = [
    col for col in numerical_cols
    if col not in missing_indicators
]

boolean_cols += missing_indicators

In [156]:
numerical_cols

['horsepower', 'mileage', 'owner_count', 'year']

In [157]:
categorical_cols

['city',
 'dealer_zip',
 'engine_type',
 'frame_damaged',
 'fuel_type',
 'has_accidents',
 'make_name',
 'model_name',
 'salvage',
 'transmission',
 'trim_name',
 'wheel_system']

In [158]:
boolean_cols

['is_new', 'owner_count_missing', 'mileage_missing', 'horsepower_missing']

In [159]:
valid_prices[numerical_cols].head(20)

,horsepower,mileage,owner_count,year
0,177.0,7.0,NaN,2019
1,246.0,8.0,NaN,2020
2,305.0,NaN,3.0,2016
3,340.0,11.0,NaN,2020
4,246.0,7.0,NaN,2020
5,247.0,12.0,NaN,2020
6,186.0,14.0,NaN,2019
7,247.0,11.0,NaN,2020
8,246.0,8.0,NaN,2020
9,296.0,254.0,NaN,2020


In [160]:
valid_prices[categorical_cols].head(5)

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,make_name,model_name,salvage,transmission,trim_name,wheel_system
0,Bayamon,960,I4,Not Reported,Gasoline,Not Reported,Jeep,Renegade,Not Reported,A,Latitude FWD,FWD
1,San Juan,922,I4,Not Reported,Gasoline,Not Reported,Land Rover,Discovery Sport,Not Reported,A,S AWD,AWD
2,Guaynabo,969,H4,False,Gasoline,False,Subaru,WRX STI,False,M,Base,AWD
3,San Juan,922,V6,Not Reported,Gasoline,Not Reported,Land Rover,Discovery,Not Reported,A,V6 HSE AWD,AWD
4,San Juan,922,I4,Not Reported,Gasoline,Not Reported,Land Rover,Discovery Sport,Not Reported,A,S AWD,AWD


In [161]:
valid_prices[boolean_cols].head(5)

,is_new,owner_count_missing,mileage_missing,horsepower_missing
0,True,1,0,0
1,True,1,0,0
2,False,0,1,0
3,True,1,0,0
4,True,1,0,0


In [162]:
valid_prices[categorical_cols].nunique().sort_values(ascending=False)

dealer_zip       10411
trim_name         9057
city              4687
model_name        1427
make_name          100
engine_type         41
fuel_type            9
wheel_system         6
transmission         5
frame_damaged        3
has_accidents        3
salvage              3
dtype: int64

In [163]:
valid_prices["model_name"].value_counts().head(20)

model_name
F-150             119053
1500               63458
Silverado 1500     63176
Escape             52671
Equinox            50506
Explorer           46466
Grand Cherokee     40429
Rogue              39546
Camry              37558
Fusion             35634
Altima             32125
Corolla            30429
Accord             29035
CR-V               28888
Edge               28130
RAV4               27756
Cherokee           27283
Tucson             26701
Civic              26223
Malibu             25998
Name: count, dtype: int64

THIS CAME AFTER THE MODEL DURING ERROR ANAYSIS

In [164]:
valid_prices.loc[[488893, 59100,1478429]]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
488893,Hinsdale,60521,V10,False,Gasoline,True,605.0,False,Porsche,338.0,Carrera GT,3.0,984900.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0
59100,Parsippany-Troy Hills,7054,V10,False,Gasoline,False,605.0,False,Porsche,2300.0,Carrera GT,7.0,819000.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0
1478429,West Chicago,60185,V8,False,Gasoline,False,NaN,False,Porsche,1034.0,918 Spyder,1.0,1225800.0,False,A,Unknown,Unknown,2015,0,0,1


In [165]:
valid_prices.loc[valid_prices["model_name"] == "Carrera GT"]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
59100,Parsippany-Troy Hills,7054,V10,False,Gasoline,False,605.0,False,Porsche,2300.0,Carrera GT,7.0,819000.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0
488893,Hinsdale,60521,V10,False,Gasoline,True,605.0,False,Porsche,338.0,Carrera GT,3.0,984900.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0
2077623,Carrollton,75006,V10,False,Gasoline,True,605.0,False,Porsche,4258.0,Carrera GT,1.0,604991.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0
2239481,Houston,77090,V10,False,Gasoline,False,605.0,False,Porsche,1901.0,Carrera GT,4.0,829991.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0


In [166]:
valid_prices.loc[valid_prices["model_name"] == "918 Spyder"]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
1478429,West Chicago,60185,V8,False,Gasoline,False,NaN,False,Porsche,1034.0,918 Spyder,1.0,1225800.0,False,A,Unknown,Unknown,2015,0,0,1
1838955,Saint Louis,63143,V8,False,Gasoline,False,887.0,False,Porsche,175.0,918 Spyder,1.0,1499999.0,False,A,Roadster,AWD,2015,0,0,0
2489699,Beverly Hills,90211,V8,False,Gasoline,False,887.0,False,Porsche,1137.0,918 Spyder,1.0,1385900.0,False,A,Roadster with Weissach,AWD,2015,0,0,0
2508115,Palm Springs,92264,V8,False,Gasoline,False,NaN,False,Porsche,382.0,918 Spyder,1.0,1285000.0,False,A,Unknown,Unknown,2015,0,0,1


In [167]:
valid_prices.loc[
    (valid_prices["model_name"] == "7 Series") &
    (valid_prices["trim_name"] == "750iL RWD")
]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
278528,Levittown,19057,V12,False,Gasoline,False,322.0,False,BMW,125555.0,7 Series,5.0,3995.0,True,A,750iL RWD,RWD,1997,0,0,0
695721,Portsmouth,23701,V12,False,Gasoline,False,322.0,False,BMW,NaN,7 Series,6.0,4599.0,False,A,750iL RWD,RWD,1998,0,1,0
941381,Greenville,29607,V12,False,Gasoline,False,296.0,False,BMW,NaN,7 Series,NaN,2500.0,False,A,750iL RWD,RWD,1988,1,1,0
941607,Greenville,29607,V12,False,Gasoline,False,300.0,False,BMW,NaN,7 Series,2.0,4500.0,False,A,750iL RWD,RWD,1989,0,1,0
2217415,Las Vegas,89139,V12,False,Gasoline,False,322.0,False,BMW,121043.0,7 Series,4.0,1750000.0,False,A,750iL RWD,RWD,1996,0,0,0
2490493,Costa Mesa,92627,V12,False,Gasoline,False,326.0,False,BMW,80227.0,7 Series,3.0,18900.0,False,A,750iL RWD,RWD,2000,0,0,0


In [ ]:
comparison_cols = [
    "horsepower",
    "mileage",
    "owner_count",
    "year",
    "is_new",
    "city",
    "engine_type",
    "frame_damaged",
    "fuel_type",
    "has_accidents",
    "make_name",
    "model_name",
    "salvage",
    "transmission",
    "trim_name",
    "wheel_system"
]


In [ ]:
train_check = X_train[comparison_cols].copy()
train_check["price"] = y_train
train_check["split"] = "train"

val_check = X_t[comparison_cols].copy()
val_check["price"] = y_train
val_check["split"] = "train"

THIS IS WHERE THE ERROR ANALYSIS STOPS

In [168]:
model_counts = valid_prices["model_name"].value_counts()
for threshold in [10,50,100,500]:
   rare_models = (model_counts < threshold).sum()
   print(f"Models with fewer than {threshold} listings: {rare_models}")

Models with fewer than 10 listings: 432
Models with fewer than 50 listings: 694
Models with fewer than 100 listings: 804
Models with fewer than 500 listings: 1036


In [169]:
model_counts.head(20)

model_name
F-150             119053
1500               63458
Silverado 1500     63176
Escape             52671
Equinox            50506
Explorer           46466
Grand Cherokee     40429
Rogue              39546
Camry              37558
Fusion             35634
Altima             32125
Corolla            30429
Accord             29035
CR-V               28888
Edge               28130
RAV4               27756
Cherokee           27283
Tucson             26701
Civic              26223
Malibu             25998
Name: count, dtype: int64

In [170]:
threshold = 500

rare_models = model_counts[model_counts < threshold].index

rare_row_mask = valid_prices["model_name"].isin(rare_models)

rare_row_percentage = rare_row_mask.mean() *100

print(f"Rows affected: {rare_row_mask.sum():,}")
print(f"Percentage affected: {rare_row_percentage:.2f}%")

Rows affected: 72,296
Percentage affected: 2.72%


In [171]:
threshold = 100

rare_models = model_counts[model_counts < threshold].index

rare_row_mask = valid_prices["model_name"].isin(rare_models)

rare_row_percentage = rare_row_mask.mean() *100

print(f"Rows affected: {rare_row_mask.sum():,}")
print(f"Percentage affected: {rare_row_percentage:.2f}%")

Rows affected: 15,480
Percentage affected: 0.58%


In [172]:
model_counts.value_counts().sort_index()

count
1         157
2          77
3          56
4          45
5          24
         ... 
50506       1
52671       1
63176       1
63458       1
119053      1
Name: count, Length: 644, dtype: int64

In [173]:
models_once = model_counts[model_counts == 1]

models_once

model_name
C8                 1
250 GT             1
Manhattan          1
Vigor              1
Lagonda            1
                  ..
Tribute Hybrid     1
Falcon Futura      1
CCXR Trevita       1
TR3                1
C/K 1000 Series    1
Name: count, Length: 157, dtype: int64

In [174]:
len(models_once)

157

In [175]:
from sklearn.model_selection import train_test_split

X = valid_prices.drop(columns=["price"])
y = valid_prices["price"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

In [176]:
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(2130205, 20) (266276, 20) (266276, 20)
(2130205,) (266276,) (266276,)


In [177]:
threshold = 10

model_counts = X_train["model_name"].value_counts()
common_models = model_counts[model_counts >= threshold].index

In [178]:
X_train_10 = X_train.copy()
X_val_10 = X_val.copy()
X_test_10 = X_test.copy()

In [179]:
for dataset in [X_train_10, X_val_10, X_test_10]:
    dataset["model_name"] = dataset["model_name"].where(
        dataset["model_name"].isin(common_models),
        "Rare"
    )

In [180]:
X_train_10["model_name"].value_counts()

model_name
F-150             95199
1500              50765
Silverado 1500    50464
Escape            42165
Equinox           40382
                  ...  
500-Class            10
Revero               10
Toronado             10
914                  10
Superamerica         10
Name: count, Length: 956, dtype: int64

In [181]:
model_counts_10 = X_train_10["model_name"].value_counts()

print("Rare rows:", model_counts_10["Rare"])
print("Smallest retained model:", model_counts_10.drop("Rare").min())

Rare rows: 1386
Smallest retained model: 10


In [182]:
X_train_10[categorical_cols].nunique().sort_values(ascending=False)

dealer_zip       10194
trim_name         8771
city              4674
model_name         956
make_name           98
engine_type         41
fuel_type            9
wheel_system         6
transmission         5
frame_damaged        3
has_accidents        3
salvage              3
dtype: int64

In [183]:
for dataset in [X_train_10, X_val_10, X_test_10]:
    dataset.drop(columns=["dealer_zip"], inplace=True)

In [184]:
categorical_cols.remove("dealer_zip")

In [185]:
X_train_10["trim_name"].value_counts()

trim_name
Unknown                               82430
SE FWD                                58150
XLT SuperCrew 4WD                     31277
S FWD                                 28733
LT FWD                                26547
                                      ...  
Lariat SB                                 1
XLT Ext. Cab 4WD                          1
Work Truck Crew Cab SB                    1
4 Dr Regency Special Edition Sedan        1
2 Dr LS Standard Cab LB HD                1
Name: count, Length: 8771, dtype: int64

In [186]:
(X_train_10["trim_name"] == "Unknown").mean() * 100

np.float64(3.8695806272166298)

In [187]:
categorical_cols


['city',
 'engine_type',
 'frame_damaged',
 'fuel_type',
 'has_accidents',
 'make_name',
 'model_name',
 'salvage',
 'transmission',
 'trim_name',
 'wheel_system']

In [188]:
vehicle_config = (
    X_train_10["make_name"].astype(str)
    + " | "
    + X_train_10["model_name"].astype(str)
    + " | "
    + X_train_10["trim_name"].astype(str)
)

In [189]:
vehicle_config.nunique()

14460

In [190]:
vehicle_config.value_counts()

Ford | F-150 | XLT SuperCrew 4WD                             28152
Ford | Escape | SE AWD                                       10837
Ford | Escape | SE FWD                                       10725
Ford | F-150 | Lariat SuperCrew 4WD                          10574
Chevrolet | Malibu | LT FWD                                  10179
                                                             ...  
Ford | F-150 | Lariat SB                                         1
Ford | F-150 | XLT Ext. Cab 4WD                                  1
GMC | Sierra 2500HD | Work Truck Crew Cab SB                     1
Oldsmobile | Rare | 4 Dr Regency Special Edition Sedan           1
Chevrolet | Silverado 2500HD | 2 Dr LS Standard Cab LB HD        1
Name: count, Length: 14460, dtype: int64

In [191]:
vehicle_config.value_counts() < 10

Ford | F-150 | XLT SuperCrew 4WD                             False
Ford | Escape | SE AWD                                       False
Ford | Escape | SE FWD                                       False
Ford | F-150 | Lariat SuperCrew 4WD                          False
Chevrolet | Malibu | LT FWD                                  False
                                                             ...  
Ford | F-150 | Lariat SB                                      True
Ford | F-150 | XLT Ext. Cab 4WD                               True
GMC | Sierra 2500HD | Work Truck Crew Cab SB                  True
Oldsmobile | Rare | 4 Dr Regency Special Edition Sedan        True
Chevrolet | Silverado 2500HD | 2 Dr LS Standard Cab LB HD     True
Name: count, Length: 14460, dtype: bool

In [192]:
(vehicle_config.value_counts() < 10).sum()

np.int64(6667)

Completed:

Cleaned master dataset: valid_prices
Classified numerical, categorical, and boolean columns
Created 80/10/10 train/validation/test split
Grouped model_name values with fewer than 10 training examples into "Rare"
Marked dealer_zip for exclusion from Version 1
Decided to keep raw trim_name and encode it sparsely

In [193]:
X_train_10

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,5.0,False,A,SE FWD,FWD,2013,0,0,0
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,3.0,False,A,XLE V6 AWD,AWD,2015,0,0,0
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,2.0,False,A,LT RWD,4X2,2017,0,0,0
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,NaN,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,1,0,0
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,1.0,False,A,1.4T S FWD,FWD,2017,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110281,South Easton,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,310.0,False,Ford,69870.0,Expedition,2.0,False,A,XLT 4WD,4WD,2014,0,0,0
1693150,Galion,V6,False,Gasoline,False,375.0,True,Ford,2.0,F-150,NaN,False,A,XLT SuperCrew LB 4WD,4WD,2020,1,0,0
2356812,Corpus Christi,V8,Not Reported,Gasoline,Not Reported,395.0,True,RAM,10.0,1500,NaN,Not Reported,A,Lone Star Crew Cab RWD,4X2,2020,1,0,0
2229564,The Woodlands,I6,False,Gasoline,False,335.0,False,BMW,2200.0,X7,NaN,False,A,xDrive40i AWD,AWD,2020,1,0,0


In [194]:
X_train_10['city'].value_counts()

city
Houston          29927
San Antonio      15807
Columbus         13430
Jacksonville     11730
Miami            11670
                 ...  
Dallastown           1
Lee                  1
Prince George        1
Haughton             1
Perth Amboy          1
Name: count, Length: 4674, dtype: int64

In [195]:
city_counts = X_train_10["city"].value_counts()

rare_cities = city_counts[city_counts < 10].index

rare_city_mask = X_train_10["city"].isin(rare_cities)

rare_city_percentage = rare_city_mask.mean() * 100

print(f"Rows affected: {rare_city_mask.sum():,}")
print(f"Percentage affected: {rare_city_percentage:.2f}%")

Rows affected: 2,003
Percentage affected: 0.09%


In [196]:
X_train_10.loc[rare_city_mask, "city"].nunique()

433

In [197]:
threshold = 10

city_counts = X_train_10["city"].value_counts()
common_cities = city_counts[city_counts >= threshold].index

for dataset in [X_train_10, X_val_10, X_test_10]:
    dataset["city"] = dataset["city"].where(
        dataset["city"].isin(common_cities),
        "Rare"
    )

In [198]:
city_counts_10 = X_train_10["city"].value_counts()

print("Rare rows:", city_counts_10["Rare"])
print("Smallest retained cities:", city_counts_10.drop("Rare").min())

Rare rows: 2003
Smallest retained cities: 10


In [199]:
X_train_10[categorical_cols].nunique().sort_values(ascending=False)

trim_name        8771
city             4242
model_name        956
make_name          98
engine_type        41
fuel_type           9
wheel_system        6
transmission        5
frame_damaged       3
has_accidents       3
salvage             3
dtype: int64

In [200]:
X_train_10['make_name'].value_counts()

make_name
Ford         353617
Chevrolet    258905
Toyota       171309
Nissan       150159
Honda        125380
              ...  
Kaiser            1
Humber            1
Edsel             1
Spyker            1
Bricklin          1
Name: count, Length: 98, dtype: int64

In [201]:
(X_train_10["make_name"].value_counts() < 10).sum()




np.int64(34)

In [202]:
X_train_10["make_name"].value_counts()[
    X_train_10["make_name"].value_counts() < 10
]

make_name
Sunbeam                    8
Studebaker                 6
Austin-Healey              5
Geo                        5
Packard                    4
Maybach                    4
International Harvester    4
VPG                        3
Hudson                     3
Daewoo                     3
Nash                       2
DeTomaso                   2
Bugatti                    2
Opel                       2
DeLorean                   2
Austin                     1
Jensen                     1
Pininfarina                1
Ariel                      1
Franklin                   1
Rover                      1
Koenigsegg                 1
Pagani                     1
Morris                     1
Mobility Ventures          1
Hillman                    1
Eagle                      1
DeSoto                     1
Clenet                     1
Kaiser                     1
Humber                     1
Edsel                      1
Spyker                     1
Bricklin                   1
Name

In [203]:
makes_per_model = (
    X_train
    .groupby("model_name")["make_name"]
    .nunique()
    .sort_values(ascending=False)
)

makes_per_model[makes_per_model > 1]

model_name
Deluxe            4
Coupe             3
GT                3
Pickup            3
Elite             2
Capri             2
Commander         2
Champ             2
Eight             2
Cabriolet         2
Dakota            2
LS                2
MV-1              2
NSX               2
NX                2
Neon              2
Prizm             2
Prowler           2
Ranger            2
Roadster          2
Sebring           2
Sprinter          2
Sprinter Cargo    2
Suburban          2
TC                2
Tracker           2
Viper             2
Voyager           2
210               2
240               2
Name: make_name, dtype: int64

In [204]:
ambiguous_models = makes_per_model[makes_per_model > 1].index

(
    X_train[X_train["model_name"].isin(ambiguous_models)]
    .groupby("model_name")["make_name"]
    .unique()
)

model_name
210                                [Chevrolet, Datsun]
240                             [Volvo, Mercedes-Benz]
Cabriolet                           [Audi, Volkswagen]
Capri                               [Lincoln, Mercury]
Champ                           [Studebaker, Plymouth]
Commander                           [Jeep, Studebaker]
Coupe                         [Maserati, Ford, Willys]
Dakota                                    [Dodge, RAM]
Deluxe            [Ford, Plymouth, Pontiac, Chevrolet]
Eight                               [Mercury, Bentley]
Elite                                    [Ford, Lotus]
GT                               [Ford, McLaren, Opel]
LS                                    [Lexus, Lincoln]
MV-1                          [VPG, Mobility Ventures]
NSX                                     [Acura, Honda]
NX                                     [Lexus, Nissan]
Neon                                 [Plymouth, Dodge]
Pickup                         [Toyota, Isuzu, Willys]

In [205]:
(makes_per_model > 1).sum()

np.int64(30)

In [206]:
ambiguous_models = makes_per_model[makes_per_model > 1].index

X_train["model_name"].isin(ambiguous_models).mean() * 100

np.float64(1.2639628580347901)

Some model labels are shared across manufacturers, but they represent only 1.26% of training observations. Make and model are retained as separate features for Version 1; a combined make-model interaction can be evaluated later.

In [207]:
X_train_10[categorical_cols].nunique().sort_values(ascending=False)

trim_name        8771
city             4242
model_name        956
make_name          98
engine_type        41
fuel_type           9
wheel_system        6
transmission        5
frame_damaged       3
has_accidents       3
salvage             3
dtype: int64

In [208]:
X_train_10['engine_type'].value_counts()

engine_type
I4                           972730
V6                           541725
V8                           208643
V8 Flex Fuel Vehicle          60062
Unknown                       58969
I4 Hybrid                     49918
V6 Flex Fuel Vehicle          49908
H4                            44974
I3                            35982
I6                            22335
I6 Diesel                     17248
V8 Biodiesel                  16756
Electric Motor                11252
I4 Flex Fuel Vehicle           7349
I4 Diesel                      5743
V8 Diesel                      5033
V6 Diesel                      4837
I5                             4240
H6                             3726
V6 Biodiesel                   2729
V6 Hybrid                      2300
V12                            1066
V10                            1033
I2                              708
W12                             384
V8 Hybrid                       101
V8 Compressed Natural Gas        76
H4 Hybrid       

In [209]:
(X_train_10['engine_type'].value_counts() < 10).sum()

np.int64(7)

In [210]:
engine_counts = X_train["engine_type"].value_counts(dropna=False)

engine_counts[engine_counts < 10]

engine_type
V8 Propane                   4
V6 Compressed Natural Gas    3
I3 Hybrid                    3
W8                           3
V12 Hybrid                   2
W16                          2
V10 Diesel                   2
Name: count, dtype: int64

In [211]:
pd.crosstab(
    X_train["engine_type"],
    X_train["fuel_type"],
    normalize="index"
).round(3)

fuel_type,Biodiesel,Compressed Natural Gas,Diesel,Electric,Flex Fuel Vehicle,Gasoline,Hybrid,Propane,Unknown
engine_type,,,,,,,,,
Electric Motor,0.0,0.0,0.00,1.0,0.0,0.000,0.0,0.0,0.000
H4,0.0,0.0,0.00,0.0,0.0,1.000,0.0,0.0,0.000
H4 Hybrid,0.0,0.0,0.00,0.0,0.0,0.000,1.0,0.0,0.000
H6,0.0,0.0,0.00,0.0,0.0,1.000,0.0,0.0,0.000
I2,0.0,0.0,0.00,0.0,0.0,1.000,0.0,0.0,0.000
I3,0.0,0.0,0.00,0.0,0.0,1.000,0.0,0.0,0.000
I3 Hybrid,0.0,0.0,0.00,0.0,0.0,0.000,1.0,0.0,0.000
I4,0.0,0.0,0.00,0.0,0.0,1.000,0.0,0.0,0.000
I4 Compressed Natural Gas,0.0,1.0,0.00,0.0,0.0,0.000,0.0,0.0,0.000


In [212]:
unknown_engine_fuel = (
    X_train.loc[
        X_train["engine_type"] == "Unknown",
        "fuel_type"
    ]
    .value_counts(dropna=False)
)

unknown_engine_fuel

fuel_type
Unknown     58016
Diesel        609
Gasoline      344
Name: count, dtype: int64

In [213]:
(
    X_train.loc[
        X_train["engine_type"] == "Unknown",
        "fuel_type"
    ]
    .ne("Unknown")
    .mean()
    * 100
)

np.float64(1.6161033763502857)

In [214]:
excluded_categorical_cols = ["dealer_zip", "fuel_type"]

fuel_type was excluded because engine_type already encoded fuel information for nearly all observations. Fuel type supplied additional information for only 953 training rows, so keeping both would add redundancy with negligible coverage benefit.

pd.crosstab()

In [215]:
pd.crosstab(
    index=X_train["frame_damaged"],
    columns=[
        X_train["has_accidents"],
        X_train["salvage"]
    ],
    normalize="index"
).round(3)

has_accidents  False          True        Not Reported
salvage        False   True  False   True Not Reported
frame_damaged                                         
False          0.846  0.002  0.147  0.004          0.0
True           0.467  0.001  0.522  0.009          0.0
Not Reported   0.000  0.000  0.000  0.000          1.0

In [216]:
pd.crosstab(
    index=X_train["frame_damaged"],
    columns=[
        X_train["has_accidents"],
        X_train["salvage"]
    ],
    margins=True
)

has_accidents    False          True       Not Reported      All
salvage          False  True   False  True Not Reported         
frame_damaged                                                   
False          1050905  2819  182305  5461            0  1241490
True              5649    15    6316   109            0    12089
Not Reported         0     0       0     0       876626   876626
All            1056554  2834  188621  5570       876626  2130205

In [217]:
X_train[numerical_cols].equals(
    X_train_10[numerical_cols]
)

True

In [218]:
X_train[numerical_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
horsepower,2009834.0,251.722556,91.035547,55.0,176.0,248.0,305.0,1001.0
mileage,2048130.0,34653.996563,46493.757212,0.0,9.0,17866.0,49197.0,400000.0
owner_count,1185529.0,1.531758,0.918826,1.0,1.0,1.0,2.0,19.0
year,2130205.0,2017.429571,4.333418,1915.0,2017.0,2019.0,2020.0,2021.0


In [219]:
X_train.groupby("is_new")["mileage"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
is_new,,,,,,,,
False,1209680.0,58553.84,47580.68,1.0,25104.0,41144.0,83769.0,400000.0
True,838450.0,172.32,1025.74,0.0,2.0,6.0,12.0,49651.0


In [220]:
X_train.loc[
    X_train["is_new"] == True,
    ["year", "make_name", "model_name", "mileage", "is_new"]
].nlargest(20, "mileage")

,year,make_name,model_name,mileage,is_new
1877137,2020,Ford,EcoSport,49651.0,True
1553665,2018,Chevrolet,Silverado 1500,49500.0,True
2374148,2020,Ford,Fusion,49302.0,True
590650,2019,Nissan,Pathfinder,48899.0,True
1071968,2021,Hyundai,Palisade,48890.0,True
970461,2019,RAM,5500 Chassis,47862.0,True
1735908,2019,RAM,1500,47103.0,True
2096170,2019,Jeep,Cherokee,46887.0,True
253119,2020,Cadillac,XT4,46063.0,True
2279360,2020,Ford,Expedition,46016.0,True


In [221]:
high_mileage_new = X_train.loc[
    (X_train["is_new"] == True) &
    (X_train["mileage"] > 1_000)
]

high_mileage_new.groupby("year")["mileage"].agg(
    count="count",
    median="median",
    maximum="max"
).sort_index()

,count,median,maximum
year,,,
2018,570,6448.5,49500.0
2019,3656,4619.5,48899.0
2020,27599,3191.0,49651.0
2021,257,2078.0,48890.0


In [222]:
(
    X_train["is_new"] &
    (X_train["mileage"] > 5_000)
).value_counts()

False    2123053
True        7152
Name: count, dtype: int64

In [223]:
(
    X_train["is_new"] &
    (X_train["mileage"] > 5_000)
).value_counts()

False    2123053
True        7152
Name: count, dtype: int64

In [224]:
is_new_adjusted = (
    X_train["is_new"] &
     (
         X_train["mileage"].isna() |
         (X_train["mileage"] <= 5000) 
      ) 
)

In [225]:
X_train["new_mileage_conflict"] = (
    X_train["is_new"] &
    (X_train["mileage"] > 5_000)
)

In [226]:
X_train.loc[
    X_train["is_new"] == False,
    ["year", "make_name", "model_name", "mileage"]
].nlargest(20, "mileage")

,year,make_name,model_name,mileage
755125,2001,Ford,F-250 Super Duty,400000.0
1435171,2008,Lexus,ES 350,399496.0
1746143,2006,Chevrolet,Silverado 2500HD,398823.0
51250,2011,Lincoln,Town Car,398654.0
2172617,2005,Dodge,RAM 3500,398229.0
2442168,2004,Dodge,RAM 2500,397896.0
2088332,2013,Chevrolet,Silverado 1500,397322.0
1305283,2008,Ford,F-350 Super Duty,396495.0
1757129,2010,Chevrolet,Express Cargo,396072.0
1281308,2000,GMC,C/K 2500 Series,395570.0


In [227]:
X_train["mileage"].max()

np.float64(400000.0)

In [228]:
X_train.loc[
    X_train["mileage"].idxmax(),
    ["year", "make_name", "model_name", "mileage", "is_new"]
]

year                      2001
make_name                 Ford
model_name    F-250 Super Duty
mileage               400000.0
is_new                   False
Name: 755125, dtype: object

In [229]:
X_train["owner_count"].value_counts(dropna=False).sort_index()

owner_count
1.0     778087
2.0     264339
3.0      93155
4.0      31677
5.0      11361
6.0       4197
7.0       1560
8.0        657
9.0        304
10.0        98
11.0        57
12.0        13
13.0        10
14.0         5
15.0         5
16.0         2
18.0         1
19.0         1
NaN     944676
Name: count, dtype: int64

In [230]:
X_train.loc[
    :,
    ["year", "make_name", "model_name", "trim_name", "horsepower"]
].nlargest(20, "horsepower")

,year,make_name,model_name,trim_name,horsepower
2545871,2008,Bugatti,Veyron,16.4 Coupe AWD,1001.0
810693,2008,Bugatti,Veyron,16.4 Coupe AWD,1001.0
2481937,2014,Ferrari,LaFerrari,Coupe,949.0
2489709,2015,Ferrari,LaFerrari,Coupe,949.0
158346,2015,McLaren,P1,Coupe,903.0
2174835,2014,McLaren,P1,Coupe,903.0
1838955,2015,Porsche,918 Spyder,Roadster,887.0
2489699,2015,Porsche,918 Spyder,Roadster with Weissach,887.0
955010,2018,Dodge,Challenger,SRT Demon RWD,808.0
622883,2018,Dodge,Challenger,SRT Demon RWD,808.0


In [231]:
X_train[
    ["year", "make_name", "model_name", "trim_name"]
].nsmallest(20, "year")

,year,make_name,model_name,trim_name
1368950,1915,Ford,Model T,Unknown
243827,1915,Ford,Model T,Unknown
1956154,1915,Ford,Model T,Unknown
1009005,1921,Ford,Model T,Unknown
1006881,1921,Ford,Model T,Unknown
1789996,1923,Ford,Model T,Dragster
2243609,1923,Ford,Model T,Dragster
1448234,1923,Ford,Model T,Dragster
820687,1923,Ford,Model T,Dragster
1007341,1923,Ford,Model T,Dragster


In [232]:
boolean_audit = (
    X_train[boolean_cols]
    .agg(["nunique", "sum", "mean"])
    .T
)

boolean_audit["percent_true"] = (
    boolean_audit["mean"] * 100
)

boolean_audit

,nunique,sum,mean,percent_true
is_new,2.0,908524.0,0.426496,42.649604
owner_count_missing,2.0,944676.0,0.443467,44.346718
mileage_missing,2.0,82075.0,0.038529,3.852916
horsepower_missing,2.0,120371.0,0.056507,5.650677


In [233]:
(
    X_train_10
    .groupby("is_new")[
        [
            "owner_count_missing",
            "mileage_missing",
            "horsepower_missing"
        ]
    ]
    .mean()
    .mul(100)
    .round(2)
)

,owner_count_missing,mileage_missing,horsepower_missing
is_new,,,
False,3.02,0.98,3.99
True,99.92,7.71,7.88


In [234]:
used_owner_count_missing = (
    (~X_train["is_new"]) &
    X_train["owner_count"].isna()
)

In [235]:
X_train.nunique()

city                      4674
dealer_zip               10194
engine_type                 41
frame_damaged                3
fuel_type                    9
has_accidents                3
horsepower                 455
is_new                       2
make_name                   98
mileage                 189013
model_name                1399
owner_count                 18
salvage                      3
transmission                 5
trim_name                 8771
wheel_system                 6
year                        98
owner_count_missing          2
mileage_missing              2
horsepower_missing           2
new_mileage_conflict         2
dtype: int64

In [236]:
X_train_10.nunique()

city                     4242
engine_type                41
frame_damaged               3
fuel_type                   9
has_accidents               3
horsepower                455
is_new                      2
make_name                  98
mileage                189013
model_name                956
owner_count                18
salvage                     3
transmission                5
trim_name                8771
wheel_system                6
year                       98
owner_count_missing         2
mileage_missing             2
horsepower_missing          2
dtype: int64

CHECK POINT 9 

In [237]:
X_train_10["new_mileage_conflict" ]= ((X_train_10["is_new"] == True) & (X_train_10["mileage"] > 5000))
X_val_10["new_mileage_conflict" ]= ((X_val_10["is_new"] == True) & (X_val_10["mileage"] > 5000))
X_test_10["new_mileage_conflict" ]= ((X_test_10["is_new"] == True) & (X_test_10["mileage"] > 5000))


                                  

In [238]:
X_train_10["used_owner_count_missing" ]= ((X_train_10["is_new"] == False) & (X_train_10["owner_count_missing"] ==True))
X_val_10["used_owner_count_missing" ]= ((X_val_10["is_new"] == False) & (X_val_10["owner_count_missing"] ==True))
X_test_10["used_owner_count_missing" ]= ((X_test_10["is_new"] == False) & (X_test_10["owner_count_missing"] ==True))

In [239]:
X_train_10

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,5.0,False,A,SE FWD,FWD,2013,0,0,0,False,False
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,3.0,False,A,XLE V6 AWD,AWD,2015,0,0,0,False,False
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,2.0,False,A,LT RWD,4X2,2017,0,0,0,False,False
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,NaN,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,1,0,0,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,1.0,False,A,1.4T S FWD,FWD,2017,0,0,0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110281,South Easton,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,310.0,False,Ford,69870.0,Expedition,2.0,False,A,XLT 4WD,4WD,2014,0,0,0,False,False
1693150,Galion,V6,False,Gasoline,False,375.0,True,Ford,2.0,F-150,NaN,False,A,XLT SuperCrew LB 4WD,4WD,2020,1,0,0,False,False
2356812,Corpus Christi,V8,Not Reported,Gasoline,Not Reported,395.0,True,RAM,10.0,1500,NaN,Not Reported,A,Lone Star Crew Cab RWD,4X2,2020,1,0,0,False,False
2229564,The Woodlands,I6,False,Gasoline,False,335.0,False,BMW,2200.0,X7,NaN,False,A,xDrive40i AWD,AWD,2020,1,0,0,False,True


In [240]:
X_train_10["Condition_reported"]=((X_train_10['frame_damaged'] != "Not Reported") & (X_train_10['has_accidents'] != "Not Reported") & (X_train_10['salvage'] != "Not Reported"))

In [241]:
X_val_10["Condition_reported"]=((X_val_10['frame_damaged'] != "Not Reported") & (X_val_10['has_accidents'] != "Not Reported") & (X_val_10['salvage'] != "Not Reported"))

In [242]:
X_test_10["Condition_reported"]=((X_test_10['frame_damaged'] != "Not Reported") & (X_test_10['has_accidents'] != "Not Reported") & (X_test_10['salvage'] != "Not Reported"))

In [243]:
X_train_10

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,5.0,False,A,SE FWD,FWD,2013,0,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,3.0,False,A,XLE V6 AWD,AWD,2015,0,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,2.0,False,A,LT RWD,4X2,2017,0,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,NaN,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,1,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,1.0,False,A,1.4T S FWD,FWD,2017,0,0,0,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110281,South Easton,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,310.0,False,Ford,69870.0,Expedition,2.0,False,A,XLT 4WD,4WD,2014,0,0,0,False,False,True
1693150,Galion,V6,False,Gasoline,False,375.0,True,Ford,2.0,F-150,NaN,False,A,XLT SuperCrew LB 4WD,4WD,2020,1,0,0,False,False,True
2356812,Corpus Christi,V8,Not Reported,Gasoline,Not Reported,395.0,True,RAM,10.0,1500,NaN,Not Reported,A,Lone Star Crew Cab RWD,4X2,2020,1,0,0,False,False,False
2229564,The Woodlands,I6,False,Gasoline,False,335.0,False,BMW,2200.0,X7,NaN,False,A,xDrive40i AWD,AWD,2020,1,0,0,False,True,True


In [244]:
X_train_10.loc[X_train_10['fuel_type'] == "Unknown"]

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1612000,St Charles,Unknown,Not Reported,Unknown,Not Reported,NaN,True,Volkswagen,1.0,Jetta,NaN,Not Reported,A,1.4T Comfortline FWD,Unknown,2020,1,0,1,False,False,False
2026500,Chandler,Unknown,Not Reported,Unknown,Not Reported,NaN,True,BMW,NaN,X3,NaN,Not Reported,A,Unknown,Unknown,2021,1,1,1,False,False,False
1596429,Louisville,Unknown,Not Reported,Unknown,Not Reported,NaN,True,Kia,7.0,Forte,NaN,Not Reported,A,LX FWD,Unknown,2021,1,0,1,False,False,False
948718,Milwaukee,Unknown,Not Reported,Unknown,Not Reported,NaN,True,Hyundai,2.0,Palisade,NaN,Not Reported,A,Unknown,Unknown,2021,1,0,1,False,False,False
1499902,Memphis,Unknown,Not Reported,Unknown,Not Reported,NaN,True,Mazda,1.0,CX-30,NaN,Not Reported,A,Unknown,Unknown,2021,1,0,1,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39357,Bridgeport,Unknown,Not Reported,Unknown,Not Reported,NaN,True,BMW,13.0,5 Series,NaN,Not Reported,A,Unknown,Unknown,2021,1,0,1,False,False,False
491312,Butler,Unknown,False,Unknown,False,450.0,False,Ford,239275.0,F-250 Super Duty,1.0,False,A,XLT Crew Cab 4WD,4WD,2019,0,0,0,False,False,True
2631202,Pleasanton,Unknown,False,Unknown,False,130.0,False,Nissan,32830.0,Sentra,1.0,False,A,S FWD,FWD,2019,0,0,0,False,False,True
1136308,Manson,Unknown,False,Unknown,True,189.0,False,Dodge,145742.0,Avenger,2.0,False,A,SXT FWD,FWD,2008,0,0,0,False,False,True


In [245]:
X_train_10.drop(columns = 'owner_count_missing', inplace = True)

In [246]:
X_val_10.drop(columns = 'owner_count_missing', inplace = True)

In [247]:
X_test_10.drop(columns = 'owner_count_missing', inplace = True)

In [248]:
X_train_10

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,5.0,False,A,SE FWD,FWD,2013,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,3.0,False,A,XLE V6 AWD,AWD,2015,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,2.0,False,A,LT RWD,4X2,2017,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,NaN,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,1.0,False,A,1.4T S FWD,FWD,2017,0,0,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110281,South Easton,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,310.0,False,Ford,69870.0,Expedition,2.0,False,A,XLT 4WD,4WD,2014,0,0,False,False,True
1693150,Galion,V6,False,Gasoline,False,375.0,True,Ford,2.0,F-150,NaN,False,A,XLT SuperCrew LB 4WD,4WD,2020,0,0,False,False,True
2356812,Corpus Christi,V8,Not Reported,Gasoline,Not Reported,395.0,True,RAM,10.0,1500,NaN,Not Reported,A,Lone Star Crew Cab RWD,4X2,2020,0,0,False,False,False
2229564,The Woodlands,I6,False,Gasoline,False,335.0,False,BMW,2200.0,X7,NaN,False,A,xDrive40i AWD,AWD,2020,0,0,False,True,True


In [249]:
X_train_10[numerical_cols].head(30)

,horsepower,mileage,owner_count,year
1201912,283.0,144084.0,5.0,2013
770704,270.0,65165.0,3.0,2015
1416224,355.0,58107.0,2.0,2017
250456,450.0,5.0,NaN,2020
2545573,150.0,33709.0,1.0,2017
2149151,500.0,19789.0,5.0,2005
1904730,370.0,21422.0,1.0,2017
660149,420.0,7.0,NaN,2020
1834720,270.0,13.0,NaN,2020
1024170,360.0,23621.0,1.0,2015


In [250]:
used_cars = X_train_10.loc[X_train_10["is_new"] == False]

In [251]:
used_cars = X_val_10.loc[X_val_10["is_new"] == False]

In [252]:
used_cars = X_test_10.loc[X_test_10["is_new"] == False]

In [253]:
used_owner_median= used_cars["owner_count"].median()

In [254]:
used_owner_median

np.float64(1.0)

In [255]:
new_cars = X_train_10.loc[X_train_10["is_new"] == True]

In [256]:
new_cars = X_val_10.loc[X_val_10["is_new"] == True]

In [257]:
new_cars = X_test_10.loc[X_test_10["is_new"] == True]

In [258]:
X_train_10.loc[
    (X_train_10["is_new"] == True)
    & (X_train_10["owner_count"].isna()),
    "owner_count"
] = 0

In [259]:
X_val_10.loc[
    (X_val_10["is_new"] == True)
    & (X_val_10["owner_count"].isna()),
    "owner_count"
] = 0

In [260]:
X_test_10.loc[
    (X_test_10["is_new"] == True)
    & (X_test_10["owner_count"].isna()),
    "owner_count"
] = 0

In [261]:
X_train_10.loc[
    (X_train_10["is_new"] == False)
    & (X_train_10["owner_count"].isna()),
    "owner_count"
] = used_owner_median

In [262]:
X_val_10.loc[
    (X_val_10["is_new"] == False)
    & (X_val_10["owner_count"].isna()),
    "owner_count"
] = used_owner_median

In [263]:
X_test_10.loc[
    (X_test_10["is_new"] == False)
    & (X_test_10["owner_count"].isna()),
    "owner_count"
] = used_owner_median

In [264]:
X_train_10["owner_count"].isna().sum()

np.int64(0)

In [265]:
X_val_10["owner_count"].isna().sum()

np.int64(0)

In [266]:
X_test_10["owner_count"].isna().sum()

np.int64(0)

# Feature Engineering Progress — August 6, 2026

## 1. Created `new_mileage_conflict`

Created a Boolean feature that identifies vehicles that:

- are listed as new;
- have more than 5,000 miles.

Applied the same rule to:

- `X_train_10`
- `X_val_10`
- `X_test_10`

This preserves the original `is_new` value while allowing the model to detect suspicious new-car mileage.

---

## 2. Defined `used_owner_count_missing`

Created a cleaner missing-value indicator for owner count.

It identifies vehicles that:

- are used;
- have a missing `owner_count`.

This is more meaningful than `owner_count_missing`, because a missing owner count for a new car usually means the feature is not applicable rather than genuinely unknown.

---

## 3. Created `condition_reported`

Created a Boolean feature that identifies whether all three condition fields were reported:

- `frame_damaged`
- `has_accidents`
- `salvage`

This separates:

- confirmed clean vehicles;
- vehicles whose condition information was not reported.

The three original condition columns will still remain separate.

---

## 4. Reviewed `fuel_type`

Observed that `engine_type` often already includes fuel information, such as:

- `V8 Diesel`
- `I4 Hybrid`
- `V8 Propane`

However, `fuel_type` has only nine categories and may still affect vehicle price.

### Decision

Keep both `engine_type` and `fuel_type` in the first model.

Their usefulness can be compared later using validation performance.

---

## 5. Completed `owner_count` imputation

### Training value

Calculated the median owner count using only used vehicles in `X_train_10`.

### Missing new-car owner counts

Missing owner counts for new vehicles were replaced with:

`0`

This represents zero previous owners.

### Missing used-car owner counts

Missing owner counts for used vehicles were replaced with the median learned from used training vehicles.

The same training median was reused for validation and test data to avoid leakage.

### Verification

Confirmed that `owner_count` now has zero missing values in:

- `X_train_10`
- `X_val_10`
- `X_test_10`

---

## Current Position

Owner-count preprocessing is complete.

### Next Step

Handle missing `mileage`, followed by missing `horsepower`, using replacement values learned from `X_train_10` only.

In [267]:
used_mileage_median = used_cars["mileage"].median()
new_mileage_median = new_cars["mileage"].median()

In [268]:
X_train_10.loc[
    (X_train_10["is_new"] == True)
    & (X_train_10["mileage"].isna()),
    "mileage"
] = new_mileage_median

In [269]:
X_val_10.loc[
    (X_val_10["is_new"] == True)
    & (X_val_10["mileage"].isna()),
    "mileage"
] = new_mileage_median

In [270]:
X_test_10.loc[
    (X_test_10["is_new"] == True)
    & (X_test_10["mileage"].isna()),
    "mileage"
] = new_mileage_median

In [271]:
X_train_10.loc[
    (X_train_10["is_new"] == False)
    & (X_train_10["mileage"].isna()),
    "mileage"
] = used_mileage_median

In [272]:
X_val_10.loc[
    (X_val_10["is_new"] == False)
    & (X_val_10["mileage"].isna()),
    "mileage"
] = used_mileage_median

In [273]:
X_test_10.loc[
    (X_test_10["is_new"] == False)
    & (X_test_10["mileage"].isna()),
    "mileage"
] = used_mileage_median

In [274]:
X_train_10["mileage"].isna().sum()

np.int64(0)

In [275]:
X_val_10["mileage"].isna().sum()

np.int64(0)

In [276]:
X_test_10["mileage"].isna().sum()

np.int64(0)

In [277]:
X_train_10["engine_type"].value_counts()

engine_type
I4                           972730
V6                           541725
V8                           208643
V8 Flex Fuel Vehicle          60062
Unknown                       58969
I4 Hybrid                     49918
V6 Flex Fuel Vehicle          49908
H4                            44974
I3                            35982
I6                            22335
I6 Diesel                     17248
V8 Biodiesel                  16756
Electric Motor                11252
I4 Flex Fuel Vehicle           7349
I4 Diesel                      5743
V8 Diesel                      5033
V6 Diesel                      4837
I5                             4240
H6                             3726
V6 Biodiesel                   2729
V6 Hybrid                      2300
V12                            1066
V10                            1033
I2                              708
W12                             384
V8 Hybrid                       101
V8 Compressed Natural Gas        76
H4 Hybrid       

In [278]:
engine_horsepower = (
    X_train_10
    .groupby
    ("engine_type") ["horsepower"]
    .median()
    )

In [279]:
engine_horsepower

engine_type
Electric Motor                170.0
H4                            175.0
H4 Hybrid                     160.0
H6                            315.0
I2                            170.0
I3                            150.0
I3 Hybrid                      73.0
I4                            180.0
I4 Compressed Natural Gas     113.0
I4 Diesel                     140.0
I4 Flex Fuel Vehicle          160.0
I4 Hybrid                     188.0
I5                            170.0
I5 Biodiesel                  185.0
I5 Diesel                     154.0
I6                            300.0
I6 Diesel                     370.0
I6 Hybrid                     335.0
R2                            238.0
Unknown                       345.0
V10                           512.0
V10 Diesel                    310.0
V12                           600.0
V12 Hybrid                    949.0
V6                            292.0
V6 Biodiesel                  260.0
V6 Compressed Natural Gas     260.0
V6 Diesel       

In [280]:
missing_hp = X_train_10["horsepower"].isna()

X_train_10.loc[missing_hp, "horsepower"] = (
    X_train_10.loc[missing_hp, "engine_type"]
    .map(engine_horsepower)
)
                              

In [281]:
missing_hp = X_val_10["horsepower"].isna()

X_val_10.loc[missing_hp, "horsepower"] = (
    X_val_10.loc[missing_hp, "engine_type"]
    .map(engine_horsepower)
)

In [282]:
missing_hp = X_test_10["horsepower"].isna()

X_test_10.loc[missing_hp, "horsepower"] = (
    X_test_10.loc[missing_hp, "engine_type"]
    .map(engine_horsepower)
)

In [283]:
X_train_10["horsepower"].isna().sum()

np.int64(0)

In [284]:
X_val_10["horsepower"].isna().sum()

np.int64(0)

In [285]:
X_test_10["horsepower"].isna().sum()

np.int64(0)

In [286]:
numerical_cols 

['horsepower', 'mileage', 'owner_count', 'year']

In [287]:
X_train_10[numerical_cols].isna().sum()

horsepower     0
mileage        0
owner_count    0
year           0
dtype: int64

In [288]:
X_val_10[numerical_cols].isna().sum()
X_test_10[numerical_cols].isna().sum()

horsepower     0
mileage        0
owner_count    0
year           0
dtype: int64

In [289]:
X_test_10[numerical_cols].isna().sum()

horsepower     0
mileage        0
owner_count    0
year           0
dtype: int64

In [290]:
X_val_10[numerical_cols].isna().sum()

horsepower     0
mileage        0
owner_count    0
year           0
dtype: int64

In [291]:
boolean_cols = [
    "is_new",
    "mileage_missing",
    "horsepower_missing",
    "Condition_reported",
    "new_mileage_conflict",
    "used_owner_count_missing"
]

In [292]:
X_val_10[boolean_cols].isna().sum()

is_new                      0
mileage_missing             0
horsepower_missing          0
Condition_reported          0
new_mileage_conflict        0
used_owner_count_missing    0
dtype: int64

In [293]:
X_test_10[boolean_cols].isna().sum()

is_new                      0
mileage_missing             0
horsepower_missing          0
Condition_reported          0
new_mileage_conflict        0
used_owner_count_missing    0
dtype: int64

In [294]:
X_train_10[boolean_cols].isna().sum()

is_new                      0
mileage_missing             0
horsepower_missing          0
Condition_reported          0
new_mileage_conflict        0
used_owner_count_missing    0
dtype: int64

In [295]:
X_train_10[categorical_cols].isna().sum()

city             0
engine_type      0
frame_damaged    0
fuel_type        0
has_accidents    0
make_name        0
model_name       0
salvage          0
transmission     0
trim_name        0
wheel_system     0
dtype: int64

In [296]:
X_val_10[categorical_cols].isna().sum()

city             0
engine_type      0
frame_damaged    0
fuel_type        0
has_accidents    0
make_name        0
model_name       0
salvage          0
transmission     0
trim_name        0
wheel_system     0
dtype: int64

In [297]:
X_train_10.shape


(2130205, 21)

In [298]:
X_test_10[categorical_cols].isna().sum()

city             0
engine_type      0
frame_damaged    0
fuel_type        0
has_accidents    0
make_name        0
model_name       0
salvage          0
transmission     0
trim_name        0
wheel_system     0
dtype: int64

In [299]:
X_train_10.shape

(2130205, 21)

PREPROCESSING CHnging cols to NUMERICAL MATRIX

In [300]:
numerical_cols

['horsepower', 'mileage', 'owner_count', 'year']

In [301]:
categorical_cols

['city',
 'engine_type',
 'frame_damaged',
 'fuel_type',
 'has_accidents',
 'make_name',
 'model_name',
 'salvage',
 'transmission',
 'trim_name',
 'wheel_system']

In [302]:
boolean_cols

['is_new',
 'mileage_missing',
 'horsepower_missing',
 'Condition_reported',
 'new_mileage_conflict',
 'used_owner_count_missing']

In [303]:
X_train_10[categorical_cols] = X_train_10[categorical_cols].astype(str)
X_val_10[categorical_cols] = X_val_10[categorical_cols].astype(str)
X_test_10[categorical_cols] = X_test_10[categorical_cols].astype(str)

In [304]:
X_train_10.to_parquet("..\data\processed\X_train.parquet")
X_test_10.to_parquet("..\data\processed\X_test.parquet")
X_val_10.to_parquet("..\data\processed\X_val.parquet")


y_train.to_frame("price").to_parquet("..\data\processed\y_train.parquet")
y_val.to_frame("price").to_parquet("..\data\processed\y_val.parquet")
y_test.to_frame("price").to_parquet("..\data\processed\y_test.parquet")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
C:\Users\casey\AppData\Local\Temp\ipykernel_3260\1789172165.py:1: SyntaxWarning: invalid escape sequence '\d'
  X_train_10.to_parquet("..\data\processed\X_train.parquet")
C:\Users\casey\AppData\Local\Temp\ipykernel_3260\1789172165.py:2: SyntaxWarning: invalid escape sequence '\d'
  X_test_10.to_parquet("..\data\processed\X_test.parquet")
C:\Users\casey\AppData\Local\Temp\ipykernel_3260\1789172165.

 Cleaned and split the data
 Engineered features
 Handled missing values
 Scaled numerical features
 One-hot encoded categorical features
 Passed Boolean features through
 Built the ColumnTransformer
 Trained multiple linear regression
 Predicted validation prices
 Evaluated the baseline


In [305]:
(X_train_10["model_name"] == "Carrera GT").sum()

np.int64(0)

In [306]:
X_train.loc[X_train["model_name"] == "Carrera GT"]

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing,new_mileage_conflict
2239481,Houston,77090,V10,False,Gasoline,False,605.0,False,Porsche,1901.0,Carrera GT,4.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0,False
2077623,Carrollton,75006,V10,False,Gasoline,True,605.0,False,Porsche,4258.0,Carrera GT,1.0,False,M,2 Dr STD Convertible,RWD,2005,0,0,0,False


In [307]:
carrera_indices = valid_prices.loc[
    valid_prices["model_name"] == "Carrera GT"
].index

In [308]:
carrera_indices.isin(X_train.index).sum()


np.int64(2)

In [309]:
carrera_indices.isin(X_val.index).sum()

np.int64(2)

In [310]:
carrera_indices.isin(X_test.index).sum()

np.int64(0)

In [311]:
spyder_indices = valid_prices.loc[
    valid_prices["model_name"] == "918 Spyder"
].index

In [312]:
spyder_indices.isin(X_train.index).sum()


np.int64(3)

In [313]:
spyder_indices.isin(X_val.index).sum()

np.int64(1)

In [314]:

spyder_indices.isin(X_test.index).sum()

np.int64(0)

In [315]:
X_val_10.loc[
    X_val.index.intersection(carrera_indices),
    "model_name"
]

488893    Rare
59100     Rare
Name: model_name, dtype: str

In [316]:
X_val_10.loc[
    X_val.index.intersection(spyder_indices),
    "model_name"
]

1478429    Rare
Name: model_name, dtype: str

In [317]:
(X_train_10["model_name"] == "918 Spyder").sum()

np.int64(0)

In [318]:
(X_train["trim_name"] == "750iL RWD").sum()

np.int64(3)

In [319]:
y_train.loc[X_train["trim_name"] == "750iL RWD"]

2490493      18900.0
695721        4599.0
2217415    1750000.0
Name: price, dtype: float64

In [320]:
BMW_indices = valid_prices.loc[
    valid_prices["trim_name"] == "750iL RWD"
].index

In [321]:
BMW_indices.isin(X_train.index).sum()

np.int64(3)

In [322]:
BMW_indices.isin(X_val.index).sum()

np.int64(1)

In [323]:
BMW_indices.isin(X_test.index).sum()

np.int64(2)

In [324]:
bad_bmw = valid_prices.loc[
    (valid_prices["trim_name"] == "750iL RWD") &
    (valid_prices["price"] == 1750000)
]

bad_bmw

,city,dealer_zip,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,owner_count,price,salvage,transmission,trim_name,wheel_system,year,owner_count_missing,mileage_missing,horsepower_missing
2217415,Las Vegas,89139,V12,False,Gasoline,False,322.0,False,BMW,121043.0,7 Series,4.0,1750000.0,False,A,750iL RWD,RWD,1996,0,0,0


In [325]:
bad_bmw.index.isin(X_train.index)

array([ True])

## Error Analysis Summary

| Case | What we found | Root cause | Proposed fix |
|---|---|---|---|
| Mercedes E-Class | ~$1.1M labels while comparable listings were ~$57k–$61k | Likely corrupted target prices + repeated/feature-identical listings | Remove invalid targets; address repeated listings across splits |
| Land Rover LR3 | ~$700k listing vs comparable ~$7k–$11k | Likely corrupted target price | Remove invalid target |
| Ford F-250 | ~$589k listing vs comparable roughly ~$30k–$100k | Likely corrupted target price | Remove invalid target |
| Ford GT | ~$975k price appears plausible for the vehicle | Legitimate high-value vehicle | Keep |
| Carrera GT / 918 Spyder | Legitimate expensive cars were converted to `Rare` | Rare-category grouping removed critical model identity | Rework `model_name` rare grouping |
| BMW 750iL | $4.5k car predicted ~$586k; training contained same trim at suspicious $1.75M | Corrupted training label distorted learned relationship | Remove invalid training target |
| Train/Val duplicates | Feature-identical Mercedes appeared across training and validation | Random splitting allowed repeated listings across splits | Quantify and improve split/dedup strategy |

In [326]:
comparison_cols = [
    "horsepower",
    "mileage",
    "owner_count",
    "year",
    "is_new",
    "city",
    "engine_type",
    "frame_damaged",
    "fuel_type",
    "has_accidents",
    "make_name",
    "model_name",
    "salvage",
    "transmission",
    "trim_name",
    "wheel_system"
]


In [327]:
train_check = X_train[comparison_cols].copy()
train_check["price"] = y_train
train_check["split"] = "train"

val_check = X_val[comparison_cols].copy()
val_check["price"] = y_val
val_check["split"] = "val"

In [328]:
duplicate_check = pd.concat([train_check, val_check])

In [329]:
duplicate_check["signature"] = pd.util.hash_pandas_object(duplicate_check[comparison_cols], index =False)



In [335]:
duplicate_summary = duplicate_check.groupby("signature").agg(
    rows=("price", "size"),
    unique_prices =("price", "nunique"),
    splits =("split", "nunique")
)

In [336]:
conflicting_groups = duplicate_summary[
    (duplicate_summary["unique_prices"] > 1) &
    (duplicate_summary["splits"] > 1)
]

In [337]:
len(conflicting_groups)

43899

In [338]:
dup_con = duplicate_check["signature"].isin(conflicting_groups.index)

In [339]:
affected_rows = duplicate_check.loc[dup_con]

In [340]:
affected_rows["split"].value_counts()

split
train    122749
val       51105
Name: count, dtype: int64

In [341]:
price_disagreement = affected_rows.groupby("signature")["price"].agg(["min", "max"])

In [342]:
price_disagreement["price_difference"] = (
    price_disagreement["max"] - price_disagreement["min"]
)

In [343]:
price_disagreement.sort_values(
    "price_difference",
    ascending=False
).head(20)

,min,max,price_difference
signature,,,
529570364717166723,57500.0,1116711.0,1059211.0
11790576227711120734,61405.0,1116711.0,1055306.0
12490102860196558516,32368.0,425460.0,393092.0
8802012891501289400,44370.0,426450.0,382080.0
11061680824326041140,50195.0,133995.0,83800.0
5645500170338199136,71400.0,150980.0,79580.0
12720954458440950481,49999.0,126999.0,77000.0
12137279769934836999,53219.0,129971.0,76752.0
7967634248240652590,69995.0,144894.0,74899.0


In [344]:
price_disagreement["price_difference"].describe()

count    4.389900e+04
mean     2.689124e+03
std      8.410293e+03
min      1.000000e+00
25%      4.950000e+02
50%      1.480000e+03
75%      3.338500e+03
max      1.059211e+06
Name: price_difference, dtype: float64

In [345]:
price_disagreement["price_difference"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
)

0.50     1480.00
0.75     3338.50
0.90     6393.20
0.95     8842.10
0.99    16435.14
Name: price_difference, dtype: float64

What waslearned from this investigation

We can keep these findings:

We found 43,899 identical 16-feature combinations crossing train/validation with different prices.
About 51k validation rows are involved.
Most price disagreements are relatively modest.
Some are absurd, like the Mercedes million-dollar cases.
But without VIN, we cannot prove these are duplicate physical vehicles.

So the correct conclusion is:

Potential repeated/indistinguishable records exist across splits, but the dataset lacks a reliable vehicle identifier to resolve them confidently. Therefore, we will not remove these rows based solely on feature similarity.